# Gopi TEC São Luís analysis - December/2024

In [ ]:
#Download Gopi_TEC_SAVO_09_27_2024_and_Dec_2024.zip from https://doi.org/10.5281/zenodo.15453941

In [ ]:
!gdown '1oCe8-iiQdu1D9HmOfW9zNa_RKJ6OVrhO'

In [ ]:
%pwd

In [ ]:
!ls

In [ ]:
!unzip 'Gopi_TEC_SALU_09_27_2024_and_Dec_2024.zip'

In [ ]:
import pandas as pd
import numpy as np
import os
from datetime import datetime, timedelta
import glob

data_dir = '.'

std_files = glob.glob(os.path.join(data_dir, 'salu3*-2024-12-*.Std'))

std_files.sort()

dfs = []

for file_path in std_files:
    try:

        filename = os.path.basename(file_path)
        print(f"Processing {filename}")

        date_parts = filename.replace('.Std', '').split('-')
        year = int(date_parts[1])
        month = int(date_parts[2])
        day = int(date_parts[3])
        base_date = datetime(year, month, day)

        df = pd.read_csv(file_path, sep='\s+', header=None)

        df.columns = ['time_ut', 'tec', 'tec_std', 'latitude']

        def decimal_to_time(decimal_hours):
            hours = int(decimal_hours)
            minutes = int((decimal_hours - hours) * 60)
            seconds = int(((decimal_hours - hours) * 60 - minutes) * 60)
            return base_date + timedelta(hours=hours, minutes=minutes, seconds=seconds)

        df['DATETIME'] = df['time_ut'].apply(decimal_to_time)

        df['tec'] = pd.to_numeric(df['tec'].replace('-', np.nan))
        df['tec_std'] = pd.to_numeric(df['tec_std'].replace('-', np.nan))

        df = df[['DATETIME', 'tec', 'tec_std', 'latitude']]
        df.columns = ['DATETIME', 'TEC', 'TEC_STD', 'LATITUDE']

        dfs.append(df)

    except Exception as e:
        print(f"Erro ao processar o arquivo {file_path}: {e}")

if dfs:
    combined_df = pd.concat(dfs, ignore_index=True)

    combined_df = combined_df.sort_values('DATETIME')

    combined_df.to_pickle('salu_dec_2024_complete.pkl')

    print(f"DataFrame combinado criado com {len(combined_df)} linhas")
    print("Primeiras 5 linhas:")
    print(combined_df.head())

    print("\nPeríodo coberto pelos dados:")
    print(f"Início: {combined_df['DATETIME'].min()}")
    print(f"Fim: {combined_df['DATETIME'].max()}")
    print(f"Total de dias: {(combined_df['DATETIME'].max() - combined_df['DATETIME'].min()).days + 1}")

    dias_presentes = combined_df['DATETIME'].dt.date.unique()
    print(f"\nTotal de dias com dados: {len(dias_presentes)}")
    print("Dias presentes:", sorted(dias_presentes))
else:
    print("Nenhum arquivo foi processado com sucesso.")

In [ ]:
combined_df

## 18:00

In [ ]:
import pandas as pd
import numpy as np
from datetime import timedelta, time

df = pd.read_pickle('salu_dec_2024_complete.pkl')
print(f"Data loaded: {len(df)} records of {df['DATETIME'].min().date()} until {df['DATETIME'].max().date()}")

df = df.sort_values('DATETIME').reset_index(drop=True)

df_clean = df.dropna(subset=['TEC', 'TEC_STD']).copy()
print(f"Analysing {len(df_clean)} points after removing missing values")

def check_consecutive_highs(df, peak_datetime, threshold_percent=90):

    peak_idx = df[df['DATETIME'] == peak_datetime].index[0]

    peak_value = df.loc[peak_idx, 'TEC']

    threshold = peak_value * (threshold_percent / 100)

    idx_before = peak_idx - 1
    idx_after = peak_idx + 1

    valid_before = idx_before >= 0
    valid_after = idx_after < len(df)

    if valid_before and valid_after:
        values = [df.loc[idx_before, 'TEC'], peak_value, df.loc[idx_after, 'TEC']]
        stds = [df.loc[idx_before, 'TEC_STD'], df.loc[peak_idx, 'TEC_STD'], df.loc[idx_after, 'TEC_STD']]
        datetimes = [df.loc[idx_before, 'DATETIME'], peak_datetime, df.loc[idx_after, 'DATETIME']]
        all_high = all(value >= threshold for value in values)

        if all_high:
            return {'indices': [idx_before, peak_idx, idx_after],
                   'values': values,
                   'stds': stds,
                   'datetimes': datetimes}

    if valid_before and idx_before > 0:
        values = [df.loc[idx_before-1, 'TEC'], df.loc[idx_before, 'TEC'], peak_value]
        stds = [df.loc[idx_before-1, 'TEC_STD'], df.loc[idx_before, 'TEC_STD'], df.loc[peak_idx, 'TEC_STD']]
        datetimes = [df.loc[idx_before-1, 'DATETIME'], df.loc[idx_before, 'DATETIME'], peak_datetime]
        all_high = all(value >= threshold for value in values)

        if all_high:
            return {'indices': [idx_before-1, idx_before, peak_idx],
                   'values': values,
                   'stds': stds,
                   'datetimes': datetimes}

    if valid_after and idx_after < len(df) - 1:
        values = [peak_value, df.loc[idx_after, 'TEC'], df.loc[idx_after+1, 'TEC']]
        stds = [df.loc[peak_idx, 'TEC_STD'], df.loc[idx_after, 'TEC_STD'], df.loc[idx_after+1, 'TEC_STD']]
        datetimes = [peak_datetime, df.loc[idx_after, 'DATETIME'], df.loc[idx_after+1, 'DATETIME']]
        all_high = all(value >= threshold for value in values)

        if all_high:
            return {'indices': [peak_idx, idx_after, idx_after+1],
                   'values': values,
                   'stds': stds,
                   'datetimes': datetimes}

    return None

def analyze_PEAKS_at_precise_interval(hour_start, minute_start, second_start, hour_end, minute_end, second_end):

    interval_label = f"{hour_start:02d}:{minute_start:02d}:{second_start:02d}-{hour_end:02d}:{minute_end:02d}:{second_end:02d}"

    print(f"\n{'='*80}")
    print(f"ANALYSIS OF TEC PEAKS IN THE PRECISE RANGE {interval_label} UTC")
    print(f"{'='*80}")

    start_timestamps = []
    end_timestamps = []

    for day in df_clean['DATETIME'].dt.date.unique():
        start_timestamps.append(pd.Timestamp(day.year, day.month, day.day, hour_start, minute_start, second_start))
        end_timestamps.append(pd.Timestamp(day.year, day.month, day.day, hour_end, minute_end, second_end))

    df_interval = pd.DataFrame()
    for start, end in zip(start_timestamps, end_timestamps):
        day_interval = df_clean[(df_clean['DATETIME'] >= start) & (df_clean['DATETIME'] <= end)]
        df_interval = pd.concat([df_interval, day_interval])

    print(f"Filtering for times between {interval_label} UTC: {len(df_interval)} remaining points")

    if len(df_interval) == 0:
        print(f"ATTENTION: No data found in the range {interval_label} UTC!")
        return pd.DataFrame(), []

    df_interval['DATE'] = df_interval['DATETIME'].dt.date

    daily_max = df_interval.loc[df_interval.groupby('DATE')['TEC'].idxmax()]
    print(f"Unique days with data in range {interval_label}: {len(daily_max)}")

    if len(daily_max) >= 5:
        top_5_days = daily_max.nlargest(5, 'TEC').reset_index(drop=True)
    else:
        top_5_days = daily_max.sort_values('TEC', ascending=False).reset_index(drop=True)
        print(f"ATTENTION: Only {len(top_5_days)} days have data in this time range!")

    print(f"\n{len(top_5_days)} HIGHEST TEC VALUES IN THE INTERVAL {interval_label} (ONE PER DAY):")
    for idx, row in top_5_days.iterrows():
        print(f"{idx+1}. TEC: {row['TEC']:.2f} ± {row['TEC_STD']:.2f} TECU, "
              f"Date/Time: {row['DATETIME']}, Latitude: {row['LATITUDE']}")

    print(f"\nSEQUENCE ANALYSIS (Values ​​remain above 90% of peak at 3 consecutive points):")
    sequences = []

    for idx, row in top_5_days.iterrows():

        sequence = check_consecutive_highs(df_clean, row['DATETIME'], threshold_percent=90)

        if sequence:
            sequences.append({
                'peak_rank': idx + 1,
                'peak_value': row['TEC'],
                'peak_std': row['TEC_STD'],
                'peak_datetime': row['DATETIME'],
                'sequence': sequence
            })

            print(f"\nPeak #{idx+1} (TEC = {row['TEC']:.2f} ± {row['TEC_STD']:.2f}) has a sequence of 3 high values:")
            for i, (dt, val, std) in enumerate(zip(sequence['datetimes'], sequence['values'], sequence['stds'])):
                print(f"  {i+1}. {dt}: {val:.2f} ± {std:.2f} TECU ({val/row['TEC']*100:.1f}% do Peak)")
        else:
            print(f"\nPeak #{idx+1} (TEC = {row['TEC']:.2f} ± {row['TEC_STD']:.2f}) "
                  f"There is NO sequence of 3 high values ​​above 90% of the peak")

    if len(top_5_days) > 0:
        print(f"\nSUMMARY TABLE OF TEC PEAKS IN THE INTERVAL {interval_label} UTC:")
        print("-" * 100)
        print(f"{'Rank':<5}{'Data':<12}{'UTC time':<12}{'TEC':<10}{'TEC_STD':<10}{'Sequence':<15}{'Sequence Duration'}")
        print("-" * 100)

        for idx, row in top_5_days.iterrows():
            has_sequence = any(seq['peak_datetime'] == row['DATETIME'] for seq in sequences)
            sequence_info = "Sim" if has_sequence else "Não"

            duration = ""
            if has_sequence:
                for seq in sequences:
                    if seq['peak_datetime'] == row['DATETIME']:
                        first_dt = min(seq['sequence']['datetimes'])
                        last_dt = max(seq['sequence']['datetimes'])
                        duration = f"{(last_dt - first_dt).total_seconds() / 60:.1f} min"
                        break

            print(f"{idx+1:<5}{row['DATETIME'].strftime('%d/%m/%Y'):<12}{row['DATETIME'].strftime('%H:%M:%S'):<12}"
                  f"{row['TEC']:.2f}    {row['TEC_STD']:.2f}    {sequence_info:<15}{duration}")
        print("-" * 100)

    return top_5_days, sequences

def analyze_PEAKS_at_interval(hour_start, minute_start, hour_end, minute_end):

    return analyze_PEAKS_at_precise_interval(hour_start, minute_start, 0, hour_end, minute_end, 0)

print("\n\n" + "*"*40 + " ANALYSIS OF THE EXACT RANGE 17:59:00-18:01:00 UTC " + "*"*40)
top_5_exact_18h, sequences_exact_18h = analyze_PEAKS_at_precise_interval(17, 59, 55, 18, 0, 5)

print("\nAnálise concluída!")

## DECEMBER

In [ ]:
import pandas as pd
import numpy as np
from datetime import timedelta, time

df = pd.read_pickle('salu_dec_2024_complete.pkl')
print(f"Data loaded: {len(df)} records of {df['DATETIME'].min().date()} until {df['DATETIME'].max().date()}")

df = df.sort_values('DATETIME').reset_index(drop=True)

df_clean = df.dropna(subset=['TEC', 'TEC_STD']).copy()
print(f"Analysing {len(df_clean)} points after removing missing values")

def check_consecutive_highs(df, peak_datetime, threshold_percent=90):

    peak_idx = df[df['DATETIME'] == peak_datetime].index[0]

    peak_value = df.loc[peak_idx, 'TEC']

    threshold = peak_value * (threshold_percent / 100)

    idx_before = peak_idx - 1
    idx_after = peak_idx + 1

    valid_before = idx_before >= 0
    valid_after = idx_after < len(df)

    if valid_before and valid_after:
        values = [df.loc[idx_before, 'TEC'], peak_value, df.loc[idx_after, 'TEC']]
        stds = [df.loc[idx_before, 'TEC_STD'], df.loc[peak_idx, 'TEC_STD'], df.loc[idx_after, 'TEC_STD']]
        datetimes = [df.loc[idx_before, 'DATETIME'], peak_datetime, df.loc[idx_after, 'DATETIME']]
        all_high = all(value >= threshold for value in values)

        if all_high:
            return {'indices': [idx_before, peak_idx, idx_after],
                   'values': values,
                   'stds': stds,
                   'datetimes': datetimes}

    if valid_before and idx_before > 0:
        values = [df.loc[idx_before-1, 'TEC'], df.loc[idx_before, 'TEC'], peak_value]
        stds = [df.loc[idx_before-1, 'TEC_STD'], df.loc[idx_before, 'TEC_STD'], df.loc[peak_idx, 'TEC_STD']]
        datetimes = [df.loc[idx_before-1, 'DATETIME'], df.loc[idx_before, 'DATETIME'], peak_datetime]
        all_high = all(value >= threshold for value in values)

        if all_high:
            return {'indices': [idx_before-1, idx_before, peak_idx],
                   'values': values,
                   'stds': stds,
                   'datetimes': datetimes}

    if valid_after and idx_after < len(df) - 1:
        values = [peak_value, df.loc[idx_after, 'TEC'], df.loc[idx_after+1, 'TEC']]
        stds = [df.loc[peak_idx, 'TEC_STD'], df.loc[idx_after, 'TEC_STD'], df.loc[idx_after+1, 'TEC_STD']]
        datetimes = [peak_datetime, df.loc[idx_after, 'DATETIME'], df.loc[idx_after+1, 'DATETIME']]
        all_high = all(value >= threshold for value in values)

        if all_high:
            return {'indices': [peak_idx, idx_after, idx_after+1],
                   'values': values,
                   'stds': stds,
                   'datetimes': datetimes}

    return None

def analyze_top_PEAKS(top_n=200):

    print(f"\n{'='*80}")
    print(f"TOP {top_n} PEAK ANALYSIS OF TEC (ALL TIMES)")
    print(f"{'='*80}")

    df_clean['DATE'] = df_clean['DATETIME'].dt.date

    top_n_PEAKS = df_clean.nlargest(top_n, 'TEC').reset_index(drop=True)

    print(f"\nTOP {top_n} HIGHEST VALUES OF TEC (ALL TIMES):")
    for idx, row in top_n_PEAKS.iterrows():
        print(f"{idx+1}. TEC: {row['TEC']:.2f} ± {row['TEC_STD']:.2f} TECU, "
              f"Date/Time: {row['DATETIME']}, Latitude: {row['LATITUDE']}")

    print(f"\nSEQUENCE ANALYSIS (Values ​​remain above 90% of peak at 3 consecutive points):")
    sequences = []

    for idx, row in top_n_PEAKS.iterrows():

        sequence = check_consecutive_highs(df_clean, row['DATETIME'], threshold_percent=90)

        if sequence:
            sequences.append({
                'peak_rank': idx + 1,
                'peak_value': row['TEC'],
                'peak_std': row['TEC_STD'],
                'peak_datetime': row['DATETIME'],
                'sequence': sequence
            })

            print(f"\nPeak #{idx+1} (TEC = {row['TEC']:.2f} ± {row['TEC_STD']:.2f}) has a sequence of 3 high values:")
            for i, (dt, val, std) in enumerate(zip(sequence['datetimes'], sequence['values'], sequence['stds'])):
                print(f"  {i+1}. {dt}: {val:.2f} ± {std:.2f} TECU ({val/row['TEC']*100:.1f}% do Peak)")
        else:
            print(f"\nPeak #{idx+1} (TEC = {row['TEC']:.2f} ± {row['TEC_STD']:.2f}) "
                  f"There is NO sequence of 3 high values ​​above 90% of the peak")

    if len(top_n_PEAKS) > 0:
        print(f"\nSUMMARY TABLE OF TOP {top_n} TEC PEAKS:")
        print("-" * 110)
        print(f"{'Rank':<5}{'Data':<12}{'UTC time':<10}{'TEC':<10}{'TEC_STD':<10}{'Sequence':<15}{'Sequence Duration':<20}{'Horário do dia'}")
        print("-" * 110)

        for idx, row in top_n_PEAKS.iterrows():
            has_sequence = any(seq['peak_datetime'] == row['DATETIME'] for seq in sequences)
            sequence_info = "Sim" if has_sequence else "Não"

            duration = ""
            if has_sequence:
                for seq in sequences:
                    if seq['peak_datetime'] == row['DATETIME']:
                        first_dt = min(seq['sequence']['datetimes'])
                        last_dt = max(seq['sequence']['datetimes'])
                        duration = f"{(last_dt - first_dt).total_seconds() / 60:.1f} min"
                        break

            hour = row['DATETIME'].hour
            minute = row['DATETIME'].minute
            time_category = f"{hour:02d}:{minute:02d} UTC"

            print(f"{idx+1:<5}{row['DATETIME'].strftime('%d/%m/%Y'):<12}{row['DATETIME'].strftime('%H:%M'):<10}"
                  f"{row['TEC']:.2f}    {row['TEC_STD']:.2f}    {sequence_info:<15}{duration:<20}{time_category}")
        print("-" * 110)

    return top_n_PEAKS, sequences

print("\n\n" + "*"*40 + "TOP ANALYSIS - 200 PEAKS OF TEC (ALL TIMES) " + "*"*40)
top_n_PEAKS, sequences = analyze_top_PEAKS(top_n=200)

print("\nAnálise concluída!")

# IGS

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
#Download TF_IGS_TEC_maps_intersection_case_study_Dec_2024.pkl from https://doi.org/10.5281/zenodo.15453941

In [ ]:
!gdown '1T0gBaH6IyIX72T1VPJuZa2D43HCrRFVp'

In [ ]:
df_mapas_igs_2024 = pd.read_pickle('/content/TF_IGS_TEC_maps_intersection_case_study_Dec_2024.pkl')

In [ ]:
df_mapas_igs_2024

## 12/18/2024 - 18:00

In [ ]:
import pandas as pd

date = '2024-12-18'
time = '18:00:00'
timestamp_wanted = f'{date} {time}'

result = df_mapas_igs_2024[df_mapas_igs_2024['DATETIME'] == timestamp_wanted]

if result.empty:

    df_temp = df_mapas_igs_2024.copy()
    df_temp['DATETIME'] = pd.to_datetime(df_temp['DATETIME'])
    timestamp_dt = pd.to_datetime(timestamp_wanted)

    df_temp['diff'] = abs((df_temp['DATETIME'] - timestamp_dt).dt.total_seconds())
    idx_nearest = df_temp['diff'].idxmin()

    result = df_mapas_igs_2024.iloc[[idx_nearest]]
    print(f"Exact date not found. Using closest record: {df_mapas_igs_2024.iloc[idx_nearest]['DATETIME']}")

In [ ]:
result

In [ ]:
mapas_igs = np.array(result.iloc[:]['TECMAP'])

In [ ]:
np_mapas_igs = []
for i in range(len(mapas_igs)):
    np_mapas_igs.append(mapas_igs[i])
np_mapas_igs = np.array(np_mapas_igs)

In [ ]:
np_mapas_igs

In [ ]:
np_mapas_igs.shape

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80, 40),
                 lat_step: float = 2.5,
                 lon_step: float = 5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class Igs(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80, 40),
                 lat_step: float = 2.5,
                 lon_step: float = 5):
        super().__init__(extent, lat_step, lon_step)


igs_tec = Igs(extent=(-110, 0, -80, 40), lat_step=2.5, lon_step=5)
igs_points = igs_tec.find_nearest_points(lat=-2.59346, lon=-44.21248)
print(igs_points)

In [ ]:
igs_points

In [ ]:
igs_tec.tec_map = np_igs_maps[0]

print("=" * 60)
print("DETAILED BILINEAR INTERPOLATION ANALYSIS")
print("=" * 60)

print("\n1. THE 4 CLOSEST POINTS:")
for i, point in enumerate(igs_points):
    lat, lon, lat_idx, lon_idx = point
    tec_value = igs_tec.tec_map[lat_idx, lon_idx]
    print(f"   P{i+1}: ({lat:+6.1f}°, {lon:+6.1f}°) - Indexes: [{lat_idx:2d}, {lon_idx:2d}] - TEC: {tec_value:5.1f}")

target_lat = -2.59346
target_lon = -44.21248

print(f"\n2. TARGET POINT: ({target_lat:+6.5f}°, {target_lon:+6.5f}°)")

# Collect values and coordinates
values = []
for point in igs_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(igs_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in igs_points]
lons = [p[1] for p in igs_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

print(f"\n3. INTERPOLATION RECTANGLE BOUNDS:")
print(f"   Latitude:  {lat_min:+6.1f}° to {lat_max:+6.1f}° (range: {lat_max - lat_min:4.1f}°)")
print(f"   Longitude: {lon_min:+6.1f}° to {lon_max:+6.1f}° (range: {lon_max - lon_min:4.1f}°)")

# Value mapping
point_values = {}
for i, point in enumerate(igs_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]  # Bottom-left corner
f_01 = point_values[(lat_min, lon_max)]  # Bottom-right corner
f_10 = point_values[(lat_max, lon_min)]  # Top-left corner
f_11 = point_values[(lat_max, lon_max)]  # Top-right corner

print(f"\n4. TEC VALUES MAPPING:")
print(f"   f_00 (lat_min, lon_min) = ({lat_min:+6.1f}°, {lon_min:+6.1f}°) = {f_00:5.1f}")
print(f"   f_01 (lat_min, lon_max) = ({lat_min:+6.1f}°, {lon_max:+6.1f}°) = {f_01:5.1f}")
print(f"   f_10 (lat_max, lon_min) = ({lat_max:+6.1f}°, {lon_min:+6.1f}°) = {f_10:5.1f}")
print(f"   f_11 (lat_max, lon_max) = ({lat_max:+6.1f}°, {lon_max:+6.1f}°) = {f_11:5.1f}")

# Interpolation parameters calculation
t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

print(f"\n5. INTERPOLATION PARAMETERS:")
print(f"   t_x = (target_lon - lon_min) / (lon_max - lon_min)")
print(f"   t_x = ({target_lon:+6.5f} - ({lon_min:+6.1f})) / ({lon_max:+6.1f} - ({lon_min:+6.1f}))")
print(f"   t_x = {target_lon - lon_min:6.5f} / {lon_max - lon_min:4.1f} = {t_x:.6f}")
print(f"   ")
print(f"   t_y = (target_lat - lat_min) / (lat_max - lat_min)")
print(f"   t_y = ({target_lat:+6.5f} - ({lat_min:+6.1f})) / ({lat_max:+6.1f} - ({lat_min:+6.1f}))")
print(f"   t_y = {target_lat - lat_min:6.5f} / {lat_max - lat_min:4.1f} = {t_y:.6f}")

# Weight calculations
w_00 = (1 - t_x) * (1 - t_y)
w_01 = t_x * (1 - t_y)
w_10 = (1 - t_x) * t_y
w_11 = t_x * t_y

print(f"\n6. INTERPOLATION WEIGHTS:")
print(f"   w_00 = (1 - t_x) * (1 - t_y) = (1 - {t_x:.6f}) * (1 - {t_y:.6f}) = {w_00:.6f}")
print(f"   w_01 = t_x * (1 - t_y)       = {t_x:.6f} * (1 - {t_y:.6f})       = {w_01:.6f}")
print(f"   w_10 = (1 - t_x) * t_y       = (1 - {t_x:.6f}) * {t_y:.6f}       = {w_10:.6f}")
print(f"   w_11 = t_x * t_y             = {t_x:.6f} * {t_y:.6f}             = {w_11:.6f}")
print(f"   Sum of weights = {w_00 + w_01 + w_10 + w_11:.6f} (should be 1.0)")

# Final interpolation
interpolated_value = w_00 * f_00 + w_01 * f_01 + w_10 * f_10 + w_11 * f_11

print(f"\n7. FINAL CALCULATION:")
print(f"   Value = w_00*f_00 + w_01*f_01 + w_10*f_10 + w_11*f_11")
print(f"   Value = {w_00:.6f}*{f_00:5.1f} + {w_01:.6f}*{f_01:5.1f} + {w_10:.6f}*{f_10:5.1f} + {w_11:.6f}*{f_11:5.1f}")
print(f"   Value = {w_00*f_00:8.4f} + {w_01*f_01:8.4f} + {w_10*f_10:8.4f} + {w_11*f_11:8.4f}")
print(f"   Value = {interpolated_value:.6f}")

print(f"\n8. TARGET POINT RELATIVE POSITION:")
print(f"   In X direction (longitude): {t_x*100:.2f}% of the way from {lon_min}° to {lon_max}°")
print(f"   In Y direction (latitude):  {t_y*100:.2f}% of the way from {lat_min}° to {lat_max}°")

print(f"\n{'='*60}")
print(f"FINAL RESULT: TEC = {interpolated_value:.4f}")
print(f"{'='*60}")

## 12/22/2024 - 18:00

In [ ]:
import pandas as pd

date = '2024-12-22'
time = '18:00:00'
timestamp_wanted = f'{date} {time}'

result = df_mapas_igs_2024[df_mapas_igs_2024['DATETIME'] == timestamp_wanted]

if result.empty:
    df_temp = df_mapas_igs_2024.copy()
    df_temp['DATETIME'] = pd.to_datetime(df_temp['DATETIME'])
    timestamp_dt = pd.to_datetime(timestamp_wanted)

    df_temp['diff'] = abs((df_temp['DATETIME'] - timestamp_dt).dt.total_seconds())
    idx_nearest = df_temp['diff'].idxmin()

    result = df_mapas_igs_2024.iloc[[idx_nearest]]
    print(f"Exact date not found. Using closest record: {df_mapas_igs_2024.iloc[idx_nearest]['DATETIME']}")

In [ ]:
result

In [ ]:
mapas_igs = np.array(result.iloc[:]['TECMAP'])

In [ ]:
np_mapas_igs = []
for i in range(len(mapas_igs)):
    np_mapas_igs.append(mapas_igs[i])
np_mapas_igs = np.array(np_mapas_igs)

In [ ]:
np_mapas_igs

In [ ]:
np_mapas_igs.shape

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80, 40),
                 lat_step: float = 2.5,
                 lon_step: float = 5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class Igs(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80, 40),
                 lat_step: float = 2.5,
                 lon_step: float = 5):
        super().__init__(extent, lat_step, lon_step)


igs_tec = Igs(extent=(-110, 0, -80, 40), lat_step=2.5, lon_step=5)
igs_points = igs_tec.find_nearest_points(lat=-2.59346, lon=-44.21248)
print(igs_points)

In [ ]:
igs_points

In [ ]:
igs_tec.tec_map = np_igs_maps[0]

print("=" * 60)
print("DETAILED BILINEAR INTERPOLATION ANALYSIS")
print("=" * 60)

print("\n1. THE 4 CLOSEST POINTS:")
for i, point in enumerate(igs_points):
    lat, lon, lat_idx, lon_idx = point
    tec_value = igs_tec.tec_map[lat_idx, lon_idx]
    print(f"   P{i+1}: ({lat:+6.1f}°, {lon:+6.1f}°) - Indexes: [{lat_idx:2d}, {lon_idx:2d}] - TEC: {tec_value:5.1f}")

target_lat = -2.59346
target_lon = -44.21248

print(f"\n2. TARGET POINT: ({target_lat:+6.5f}°, {target_lon:+6.5f}°)")

# Collect values and coordinates
values = []
for point in igs_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(igs_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in igs_points]
lons = [p[1] for p in igs_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

print(f"\n3. INTERPOLATION RECTANGLE BOUNDS:")
print(f"   Latitude:  {lat_min:+6.1f}° to {lat_max:+6.1f}° (range: {lat_max - lat_min:4.1f}°)")
print(f"   Longitude: {lon_min:+6.1f}° to {lon_max:+6.1f}° (range: {lon_max - lon_min:4.1f}°)")

# Value mapping
point_values = {}
for i, point in enumerate(igs_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]  # Bottom-left corner
f_01 = point_values[(lat_min, lon_max)]  # Bottom-right corner
f_10 = point_values[(lat_max, lon_min)]  # Top-left corner
f_11 = point_values[(lat_max, lon_max)]  # Top-right corner

print(f"\n4. TEC VALUES MAPPING:")
print(f"   f_00 (lat_min, lon_min) = ({lat_min:+6.1f}°, {lon_min:+6.1f}°) = {f_00:5.1f}")
print(f"   f_01 (lat_min, lon_max) = ({lat_min:+6.1f}°, {lon_max:+6.1f}°) = {f_01:5.1f}")
print(f"   f_10 (lat_max, lon_min) = ({lat_max:+6.1f}°, {lon_min:+6.1f}°) = {f_10:5.1f}")
print(f"   f_11 (lat_max, lon_max) = ({lat_max:+6.1f}°, {lon_max:+6.1f}°) = {f_11:5.1f}")

# Interpolation parameters calculation
t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

print(f"\n5. INTERPOLATION PARAMETERS:")
print(f"   t_x = (target_lon - lon_min) / (lon_max - lon_min)")
print(f"   t_x = ({target_lon:+6.5f} - ({lon_min:+6.1f})) / ({lon_max:+6.1f} - ({lon_min:+6.1f}))")
print(f"   t_x = {target_lon - lon_min:6.5f} / {lon_max - lon_min:4.1f} = {t_x:.6f}")
print(f"   ")
print(f"   t_y = (target_lat - lat_min) / (lat_max - lat_min)")
print(f"   t_y = ({target_lat:+6.5f} - ({lat_min:+6.1f})) / ({lat_max:+6.1f} - ({lat_min:+6.1f}))")
print(f"   t_y = {target_lat - lat_min:6.5f} / {lat_max - lat_min:4.1f} = {t_y:.6f}")

# Weight calculations
w_00 = (1 - t_x) * (1 - t_y)
w_01 = t_x * (1 - t_y)
w_10 = (1 - t_x) * t_y
w_11 = t_x * t_y

print(f"\n6. INTERPOLATION WEIGHTS:")
print(f"   w_00 = (1 - t_x) * (1 - t_y) = (1 - {t_x:.6f}) * (1 - {t_y:.6f}) = {w_00:.6f}")
print(f"   w_01 = t_x * (1 - t_y)       = {t_x:.6f} * (1 - {t_y:.6f})       = {w_01:.6f}")
print(f"   w_10 = (1 - t_x) * t_y       = (1 - {t_x:.6f}) * {t_y:.6f}       = {w_10:.6f}")
print(f"   w_11 = t_x * t_y             = {t_x:.6f} * {t_y:.6f}             = {w_11:.6f}")
print(f"   Sum of weights = {w_00 + w_01 + w_10 + w_11:.6f} (should be 1.0)")

# Final interpolation
interpolated_value = w_00 * f_00 + w_01 * f_01 + w_10 * f_10 + w_11 * f_11

print(f"\n7. FINAL CALCULATION:")
print(f"   Value = w_00*f_00 + w_01*f_01 + w_10*f_10 + w_11*f_11")
print(f"   Value = {w_00:.6f}*{f_00:5.1f} + {w_01:.6f}*{f_01:5.1f} + {w_10:.6f}*{f_10:5.1f} + {w_11:.6f}*{f_11:5.1f}")
print(f"   Value = {w_00*f_00:8.4f} + {w_01*f_01:8.4f} + {w_10*f_10:8.4f} + {w_11*f_11:8.4f}")
print(f"   Value = {interpolated_value:.6f}")

print(f"\n8. TARGET POINT RELATIVE POSITION:")
print(f"   In X direction (longitude): {t_x*100:.2f}% of the way from {lon_min}° to {lon_max}°")
print(f"   In Y direction (latitude):  {t_y*100:.2f}% of the way from {lat_min}° to {lat_max}°")

print(f"\n{'='*60}")
print(f"FINAL RESULT: TEC = {interpolated_value:.4f}")
print(f"{'='*60}")

## 12/19/2024 - 18:00

In [ ]:
import pandas as pd

date = '2024-12-19'
time = '18:00:00'
timestamp_wanted = f'{date} {time}'

result = df_mapas_igs_2024[df_mapas_igs_2024['DATETIME'] == timestamp_wanted]

if result.empty:
    df_temp = df_mapas_igs_2024.copy()
    df_temp['DATETIME'] = pd.to_datetime(df_temp['DATETIME'])
    timestamp_dt = pd.to_datetime(timestamp_wanted)

    df_temp['diff'] = abs((df_temp['DATETIME'] - timestamp_dt).dt.total_seconds())
    idx_nearest = df_temp['diff'].idxmin()

    result = df_mapas_igs_2024.iloc[[idx_nearest]]
    print(f"Exact date not found. Using closest record: {df_mapas_igs_2024.iloc[idx_nearest]['DATETIME']}")

In [ ]:
result

In [ ]:
mapas_igs = np.array(result.iloc[:]['TECMAP'])

In [ ]:
np_mapas_igs = []
for i in range(len(mapas_igs)):
    np_mapas_igs.append(mapas_igs[i])
np_mapas_igs = np.array(np_mapas_igs)

In [ ]:
np_mapas_igs

In [ ]:
np_mapas_igs.shape

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80, 40),
                 lat_step: float = 2.5,
                 lon_step: float = 5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class Igs(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80, 40),
                 lat_step: float = 2.5,
                 lon_step: float = 5):
        super().__init__(extent, lat_step, lon_step)


igs_tec = Igs(extent=(-110, 0, -80, 40), lat_step=2.5, lon_step=5)
igs_points = igs_tec.find_nearest_points(lat=-2.59346, lon=-44.21248)
print(igs_points)

In [ ]:
igs_points

In [ ]:
igs_tec.tec_map = np_igs_maps[0]

print("=" * 60)
print("DETAILED BILINEAR INTERPOLATION ANALYSIS")
print("=" * 60)

print("\n1. THE 4 CLOSEST POINTS:")
for i, point in enumerate(igs_points):
    lat, lon, lat_idx, lon_idx = point
    tec_value = igs_tec.tec_map[lat_idx, lon_idx]
    print(f"   P{i+1}: ({lat:+6.1f}°, {lon:+6.1f}°) - Indexes: [{lat_idx:2d}, {lon_idx:2d}] - TEC: {tec_value:5.1f}")

target_lat = -2.59346
target_lon = -44.21248

print(f"\n2. TARGET POINT: ({target_lat:+6.5f}°, {target_lon:+6.5f}°)")

# Collect values and coordinates
values = []
for point in igs_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(igs_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in igs_points]
lons = [p[1] for p in igs_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

print(f"\n3. INTERPOLATION RECTANGLE BOUNDS:")
print(f"   Latitude:  {lat_min:+6.1f}° to {lat_max:+6.1f}° (range: {lat_max - lat_min:4.1f}°)")
print(f"   Longitude: {lon_min:+6.1f}° to {lon_max:+6.1f}° (range: {lon_max - lon_min:4.1f}°)")

# Value mapping
point_values = {}
for i, point in enumerate(igs_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]  # Bottom-left corner
f_01 = point_values[(lat_min, lon_max)]  # Bottom-right corner
f_10 = point_values[(lat_max, lon_min)]  # Top-left corner
f_11 = point_values[(lat_max, lon_max)]  # Top-right corner

print(f"\n4. TEC VALUES MAPPING:")
print(f"   f_00 (lat_min, lon_min) = ({lat_min:+6.1f}°, {lon_min:+6.1f}°) = {f_00:5.1f}")
print(f"   f_01 (lat_min, lon_max) = ({lat_min:+6.1f}°, {lon_max:+6.1f}°) = {f_01:5.1f}")
print(f"   f_10 (lat_max, lon_min) = ({lat_max:+6.1f}°, {lon_min:+6.1f}°) = {f_10:5.1f}")
print(f"   f_11 (lat_max, lon_max) = ({lat_max:+6.1f}°, {lon_max:+6.1f}°) = {f_11:5.1f}")

# Interpolation parameters calculation
t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

print(f"\n5. INTERPOLATION PARAMETERS:")
print(f"   t_x = (target_lon - lon_min) / (lon_max - lon_min)")
print(f"   t_x = ({target_lon:+6.5f} - ({lon_min:+6.1f})) / ({lon_max:+6.1f} - ({lon_min:+6.1f}))")
print(f"   t_x = {target_lon - lon_min:6.5f} / {lon_max - lon_min:4.1f} = {t_x:.6f}")
print(f"   ")
print(f"   t_y = (target_lat - lat_min) / (lat_max - lat_min)")
print(f"   t_y = ({target_lat:+6.5f} - ({lat_min:+6.1f})) / ({lat_max:+6.1f} - ({lat_min:+6.1f}))")
print(f"   t_y = {target_lat - lat_min:6.5f} / {lat_max - lat_min:4.1f} = {t_y:.6f}")

# Weight calculations
w_00 = (1 - t_x) * (1 - t_y)
w_01 = t_x * (1 - t_y)
w_10 = (1 - t_x) * t_y
w_11 = t_x * t_y

print(f"\n6. INTERPOLATION WEIGHTS:")
print(f"   w_00 = (1 - t_x) * (1 - t_y) = (1 - {t_x:.6f}) * (1 - {t_y:.6f}) = {w_00:.6f}")
print(f"   w_01 = t_x * (1 - t_y)       = {t_x:.6f} * (1 - {t_y:.6f})       = {w_01:.6f}")
print(f"   w_10 = (1 - t_x) * t_y       = (1 - {t_x:.6f}) * {t_y:.6f}       = {w_10:.6f}")
print(f"   w_11 = t_x * t_y             = {t_x:.6f} * {t_y:.6f}             = {w_11:.6f}")
print(f"   Sum of weights = {w_00 + w_01 + w_10 + w_11:.6f} (should be 1.0)")

# Final interpolation
interpolated_value = w_00 * f_00 + w_01 * f_01 + w_10 * f_10 + w_11 * f_11

print(f"\n7. FINAL CALCULATION:")
print(f"   Value = w_00*f_00 + w_01*f_01 + w_10*f_10 + w_11*f_11")
print(f"   Value = {w_00:.6f}*{f_00:5.1f} + {w_01:.6f}*{f_01:5.1f} + {w_10:.6f}*{f_10:5.1f} + {w_11:.6f}*{f_11:5.1f}")
print(f"   Value = {w_00*f_00:8.4f} + {w_01*f_01:8.4f} + {w_10*f_10:8.4f} + {w_11*f_11:8.4f}")
print(f"   Value = {interpolated_value:.6f}")

print(f"\n8. TARGET POINT RELATIVE POSITION:")
print(f"   In X direction (longitude): {t_x*100:.2f}% of the way from {lon_min}° to {lon_max}°")
print(f"   In Y direction (latitude):  {t_y*100:.2f}% of the way from {lat_min}° to {lat_max}°")

print(f"\n{'='*60}")
print(f"FINAL RESULT: TEC = {interpolated_value:.4f}")
print(f"{'='*60}")

## 12/01/2024 - 18:00

In [ ]:
import pandas as pd

date = '2024-12-01'
time = '18:00:00'
timestamp_wanted = f'{date} {time}'

result = df_mapas_igs_2024[df_mapas_igs_2024['DATETIME'] == timestamp_wanted]

if result.empty:
    df_temp = df_mapas_igs_2024.copy()
    df_temp['DATETIME'] = pd.to_datetime(df_temp['DATETIME'])
    timestamp_dt = pd.to_datetime(timestamp_wanted)

    df_temp['diff'] = abs((df_temp['DATETIME'] - timestamp_dt).dt.total_seconds())
    idx_nearest = df_temp['diff'].idxmin()

    result = df_mapas_igs_2024.iloc[[idx_nearest]]
    print(f"Exact date not found. Using closest record: {df_mapas_igs_2024.iloc[idx_nearest]['DATETIME']}")

In [ ]:
result

In [ ]:
mapas_igs = np.array(result.iloc[:]['TECMAP'])

In [ ]:
np_mapas_igs = []
for i in range(len(mapas_igs)):
    np_mapas_igs.append(mapas_igs[i])
np_mapas_igs = np.array(np_mapas_igs)

In [ ]:
np_mapas_igs

In [ ]:
np_mapas_igs.shape

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80, 40),
                 lat_step: float = 2.5,
                 lon_step: float = 5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):
        """
        Função para encontrar os 4 pontos da grade mais próximos
        a uma coordenada específica
        """
        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class Igs(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80, 40),
                 lat_step: float = 2.5,
                 lon_step: float = 5):
        super().__init__(extent, lat_step, lon_step)


igs_tec = Igs(extent=(-110, 0, -80, 40), lat_step=2.5, lon_step=5)
igs_points = igs_tec.find_nearest_points(lat=-2.59346, lon=-44.21248)
print(igs_points)

In [ ]:
igs_points

In [ ]:
igs_tec.tec_map = np_igs_maps[0]

print("=" * 60)
print("DETAILED BILINEAR INTERPOLATION ANALYSIS")
print("=" * 60)

print("\n1. THE 4 CLOSEST POINTS:")
for i, point in enumerate(igs_points):
    lat, lon, lat_idx, lon_idx = point
    tec_value = igs_tec.tec_map[lat_idx, lon_idx]
    print(f"   P{i+1}: ({lat:+6.1f}°, {lon:+6.1f}°) - Indexes: [{lat_idx:2d}, {lon_idx:2d}] - TEC: {tec_value:5.1f}")

target_lat = -2.59346
target_lon = -44.21248

print(f"\n2. TARGET POINT: ({target_lat:+6.5f}°, {target_lon:+6.5f}°)")

# Collect values and coordinates
values = []
for point in igs_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(igs_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in igs_points]
lons = [p[1] for p in igs_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

print(f"\n3. INTERPOLATION RECTANGLE BOUNDS:")
print(f"   Latitude:  {lat_min:+6.1f}° to {lat_max:+6.1f}° (range: {lat_max - lat_min:4.1f}°)")
print(f"   Longitude: {lon_min:+6.1f}° to {lon_max:+6.1f}° (range: {lon_max - lon_min:4.1f}°)")

# Value mapping
point_values = {}
for i, point in enumerate(igs_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]  # Bottom-left corner
f_01 = point_values[(lat_min, lon_max)]  # Bottom-right corner
f_10 = point_values[(lat_max, lon_min)]  # Top-left corner
f_11 = point_values[(lat_max, lon_max)]  # Top-right corner

print(f"\n4. TEC VALUES MAPPING:")
print(f"   f_00 (lat_min, lon_min) = ({lat_min:+6.1f}°, {lon_min:+6.1f}°) = {f_00:5.1f}")
print(f"   f_01 (lat_min, lon_max) = ({lat_min:+6.1f}°, {lon_max:+6.1f}°) = {f_01:5.1f}")
print(f"   f_10 (lat_max, lon_min) = ({lat_max:+6.1f}°, {lon_min:+6.1f}°) = {f_10:5.1f}")
print(f"   f_11 (lat_max, lon_max) = ({lat_max:+6.1f}°, {lon_max:+6.1f}°) = {f_11:5.1f}")

# Interpolation parameters calculation
t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

print(f"\n5. INTERPOLATION PARAMETERS:")
print(f"   t_x = (target_lon - lon_min) / (lon_max - lon_min)")
print(f"   t_x = ({target_lon:+6.5f} - ({lon_min:+6.1f})) / ({lon_max:+6.1f} - ({lon_min:+6.1f}))")
print(f"   t_x = {target_lon - lon_min:6.5f} / {lon_max - lon_min:4.1f} = {t_x:.6f}")
print(f"   ")
print(f"   t_y = (target_lat - lat_min) / (lat_max - lat_min)")
print(f"   t_y = ({target_lat:+6.5f} - ({lat_min:+6.1f})) / ({lat_max:+6.1f} - ({lat_min:+6.1f}))")
print(f"   t_y = {target_lat - lat_min:6.5f} / {lat_max - lat_min:4.1f} = {t_y:.6f}")

# Weight calculations
w_00 = (1 - t_x) * (1 - t_y)
w_01 = t_x * (1 - t_y)
w_10 = (1 - t_x) * t_y
w_11 = t_x * t_y

print(f"\n6. INTERPOLATION WEIGHTS:")
print(f"   w_00 = (1 - t_x) * (1 - t_y) = (1 - {t_x:.6f}) * (1 - {t_y:.6f}) = {w_00:.6f}")
print(f"   w_01 = t_x * (1 - t_y)       = {t_x:.6f} * (1 - {t_y:.6f})       = {w_01:.6f}")
print(f"   w_10 = (1 - t_x) * t_y       = (1 - {t_x:.6f}) * {t_y:.6f}       = {w_10:.6f}")
print(f"   w_11 = t_x * t_y             = {t_x:.6f} * {t_y:.6f}             = {w_11:.6f}")
print(f"   Sum of weights = {w_00 + w_01 + w_10 + w_11:.6f} (should be 1.0)")

# Final interpolation
interpolated_value = w_00 * f_00 + w_01 * f_01 + w_10 * f_10 + w_11 * f_11

print(f"\n7. FINAL CALCULATION:")
print(f"   Value = w_00*f_00 + w_01*f_01 + w_10*f_10 + w_11*f_11")
print(f"   Value = {w_00:.6f}*{f_00:5.1f} + {w_01:.6f}*{f_01:5.1f} + {w_10:.6f}*{f_10:5.1f} + {w_11:.6f}*{f_11:5.1f}")
print(f"   Value = {w_00*f_00:8.4f} + {w_01*f_01:8.4f} + {w_10*f_10:8.4f} + {w_11*f_11:8.4f}")
print(f"   Value = {interpolated_value:.6f}")

print(f"\n8. TARGET POINT RELATIVE POSITION:")
print(f"   In X direction (longitude): {t_x*100:.2f}% of the way from {lon_min}° to {lon_max}°")
print(f"   In Y direction (latitude):  {t_y*100:.2f}% of the way from {lat_min}° to {lat_max}°")

print(f"\n{'='*60}")
print(f"FINAL RESULT: TEC = {interpolated_value:.4f}")
print(f"{'='*60}")

# Nagoya

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
#Download TF_Nagoya_TEC_maps_intersection_case_study_Dec_2024.pkl from https://doi.org/10.5281/zenodo.15453941

In [ ]:
!gdown '1CXL-btSyeELqYjLGL-B2oSnMt-QlPFUR'

In [ ]:
df_nagoya_maps_2024 = pd.read_pickle('TF_Nagoya_TEC_maps_intersection_case_study_Dec_2024.pkl')

In [ ]:
df_nagoya_maps_2024

## 12/18/2024 - 18:20

In [ ]:
import pandas as pd

date = '2024-12-18'
time = '18:20:00'
timestamp_wanted = f'{date} {time}'

result = df_nagoya_maps_2024[df_nagoya_maps_2024['DATETIME'] == timestamp_wanted]

if result.empty:
    df_temp = df_nagoya_maps_2024.copy()
    df_temp['DATETIME'] = pd.to_datetime(df_temp['DATETIME'])
    timestamp_dt = pd.to_datetime(timestamp_wanted)

    df_temp['diff'] = abs((df_temp['DATETIME'] - timestamp_dt).dt.total_seconds())
    idx_nearest = df_temp['diff'].idxmin()

    result = df_nagoya_maps_2024.iloc[[idx_nearest]]
    print(f"Exact date not found. Using closest record: {df_nagoya_maps_2024.iloc[idx_nearest]['DATETIME']}")

In [ ]:
result

In [ ]:
nagoya_maps = np.array(result.iloc[:]['TECMAP'])

In [ ]:
np_nagoya_maps = []
for i in range(len(nagoya_maps)):
    np_nagoya_maps.append(nagoya_maps[i])
np_nagoya_maps = np.array(np_nagoya_maps)

In [ ]:
np_nagoya_maps

In [ ]:
np_nagoya_maps.shape

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80.4, 40.1),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class nagoya(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80.4, 40.1),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        super().__init__(extent, lat_step, lon_step)


nagoya_tec = nagoya(extent=(-110, 0, -80.4, 40.1), lat_step=0.5, lon_step=0.5)
nagoya_points = nagoya_tec.find_nearest_points(lat=-2.59346, lon=-44.21248)
print(nagoya_points)

In [ ]:
nagoya_points

In [ ]:
nagoya_tec.tec_map = np_nagoya_maps[0]

print("=" * 60)
print("DETAILED BILINEAR INTERPOLATION ANALYSIS")
print("=" * 60)

print("\n1. THE 4 CLOSEST POINTS:")
for i, point in enumerate(nagoya_points):
    lat, lon, lat_idx, lon_idx = point
    tec_value = nagoya_tec.tec_map[lat_idx, lon_idx]
    print(f"   P{i+1}: ({lat:+6.1f}°, {lon:+6.1f}°) - Indexes: [{lat_idx:2d}, {lon_idx:2d}] - TEC: {tec_value:5.1f}")

target_lat = -2.59346
target_lon = -44.21248

print(f"\n2. TARGET POINT: ({target_lat:+6.5f}°, {target_lon:+6.5f}°)")

# Collect values and coordinates
values = []
for point in nagoya_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(nagoya_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in nagoya_points]
lons = [p[1] for p in nagoya_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

print(f"\n3. INTERPOLATION RECTANGLE BOUNDS:")
print(f"   Latitude:  {lat_min:+6.1f}° to {lat_max:+6.1f}° (range: {lat_max - lat_min:4.1f}°)")
print(f"   Longitude: {lon_min:+6.1f}° to {lon_max:+6.1f}° (range: {lon_max - lon_min:4.1f}°)")

# Value mapping
point_values = {}
for i, point in enumerate(nagoya_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]  # Bottom-left corner
f_01 = point_values[(lat_min, lon_max)]  # Bottom-right corner
f_10 = point_values[(lat_max, lon_min)]  # Top-left corner
f_11 = point_values[(lat_max, lon_max)]  # Top-right corner

print(f"\n4. TEC VALUES MAPPING:")
print(f"   f_00 (lat_min, lon_min) = ({lat_min:+6.1f}°, {lon_min:+6.1f}°) = {f_00:5.1f}")
print(f"   f_01 (lat_min, lon_max) = ({lat_min:+6.1f}°, {lon_max:+6.1f}°) = {f_01:5.1f}")
print(f"   f_10 (lat_max, lon_min) = ({lat_max:+6.1f}°, {lon_min:+6.1f}°) = {f_10:5.1f}")
print(f"   f_11 (lat_max, lon_max) = ({lat_max:+6.1f}°, {lon_max:+6.1f}°) = {f_11:5.1f}")

# Interpolation parameters calculation
t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

print(f"\n5. INTERPOLATION PARAMETERS:")
print(f"   t_x = (target_lon - lon_min) / (lon_max - lon_min)")
print(f"   t_x = ({target_lon:+6.5f} - ({lon_min:+6.1f})) / ({lon_max:+6.1f} - ({lon_min:+6.1f}))")
print(f"   t_x = {target_lon - lon_min:6.5f} / {lon_max - lon_min:4.1f} = {t_x:.6f}")
print(f"   ")
print(f"   t_y = (target_lat - lat_min) / (lat_max - lat_min)")
print(f"   t_y = ({target_lat:+6.5f} - ({lat_min:+6.1f})) / ({lat_max:+6.1f} - ({lat_min:+6.1f}))")
print(f"   t_y = {target_lat - lat_min:6.5f} / {lat_max - lat_min:4.1f} = {t_y:.6f}")

# Weight calculations
w_00 = (1 - t_x) * (1 - t_y)
w_01 = t_x * (1 - t_y)
w_10 = (1 - t_x) * t_y
w_11 = t_x * t_y

print(f"\n6. INTERPOLATION WEIGHTS:")
print(f"   w_00 = (1 - t_x) * (1 - t_y) = (1 - {t_x:.6f}) * (1 - {t_y:.6f}) = {w_00:.6f}")
print(f"   w_01 = t_x * (1 - t_y)       = {t_x:.6f} * (1 - {t_y:.6f})       = {w_01:.6f}")
print(f"   w_10 = (1 - t_x) * t_y       = (1 - {t_x:.6f}) * {t_y:.6f}       = {w_10:.6f}")
print(f"   w_11 = t_x * t_y             = {t_x:.6f} * {t_y:.6f}             = {w_11:.6f}")
print(f"   Sum of weights = {w_00 + w_01 + w_10 + w_11:.6f} (should be 1.0)")

# Final interpolation
interpolated_value = w_00 * f_00 + w_01 * f_01 + w_10 * f_10 + w_11 * f_11

print(f"\n7. FINAL CALCULATION:")
print(f"   Value = w_00*f_00 + w_01*f_01 + w_10*f_10 + w_11*f_11")
print(f"   Value = {w_00:.6f}*{f_00:5.1f} + {w_01:.6f}*{f_01:5.1f} + {w_10:.6f}*{f_10:5.1f} + {w_11:.6f}*{f_11:5.1f}")
print(f"   Value = {w_00*f_00:8.4f} + {w_01*f_01:8.4f} + {w_10*f_10:8.4f} + {w_11*f_11:8.4f}")
print(f"   Value = {interpolated_value:.6f}")

print(f"\n8. TARGET POINT RELATIVE POSITION:")
print(f"   In X direction (longitude): {t_x*100:.2f}% of the way from {lon_min}° to {lon_max}°")
print(f"   In Y direction (latitude):  {t_y*100:.2f}% of the way from {lat_min}° to {lat_max}°")

print(f"\n{'='*60}")
print(f"FINAL RESULT: TEC = {interpolated_value:.4f}")
print(f"{'='*60}")

## 12/18/2024 - 18:10

In [ ]:
import pandas as pd

date = '2024-12-18'
time = '18:10:00'
timestamp_wanted = f'{date} {time}'

result = df_nagoya_maps_2024[df_nagoya_maps_2024['DATETIME'] == timestamp_wanted]

if result.empty:
    df_temp = df_nagoya_maps_2024.copy()
    df_temp['DATETIME'] = pd.to_datetime(df_temp['DATETIME'])
    timestamp_dt = pd.to_datetime(timestamp_wanted)

    df_temp['diff'] = abs((df_temp['DATETIME'] - timestamp_dt).dt.total_seconds())
    idx_nearest = df_temp['diff'].idxmin()

    result = df_nagoya_maps_2024.iloc[[idx_nearest]]
    print(f"Exact date not found. Using closest record: {df_nagoya_maps_2024.iloc[idx_nearest]['DATETIME']}")

In [ ]:
result

In [ ]:
nagoya_maps = np.array(result.iloc[:]['TECMAP'])

In [ ]:
np_nagoya_maps = []
for i in range(len(nagoya_maps)):
    np_nagoya_maps.append(nagoya_maps[i])
np_nagoya_maps = np.array(np_nagoya_maps)

In [ ]:
np_nagoya_maps

In [ ]:
np_nagoya_maps.shape

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80.4, 40.1),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class nagoya(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80.4, 40.1),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        super().__init__(extent, lat_step, lon_step)


nagoya_tec = nagoya(extent=(-110, 0, -80.4, 40.1), lat_step=0.5, lon_step=0.5)
nagoya_points = nagoya_tec.find_nearest_points(lat=-2.59346, lon=-44.21248)
print(nagoya_points)

In [ ]:
nagoya_points

In [ ]:
nagoya_tec.tec_map = np_nagoya_maps[0]

print("=" * 60)
print("DETAILED BILINEAR INTERPOLATION ANALYSIS")
print("=" * 60)

print("\n1. THE 4 CLOSEST POINTS:")
for i, point in enumerate(nagoya_points):
    lat, lon, lat_idx, lon_idx = point
    tec_value = nagoya_tec.tec_map[lat_idx, lon_idx]
    print(f"   P{i+1}: ({lat:+6.1f}°, {lon:+6.1f}°) - Indexes: [{lat_idx:2d}, {lon_idx:2d}] - TEC: {tec_value:5.1f}")

target_lat = -2.59346
target_lon = -44.21248

print(f"\n2. TARGET POINT: ({target_lat:+6.5f}°, {target_lon:+6.5f}°)")

# Collect values and coordinates
values = []
for point in nagoya_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(nagoya_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in nagoya_points]
lons = [p[1] for p in nagoya_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

print(f"\n3. INTERPOLATION RECTANGLE BOUNDS:")
print(f"   Latitude:  {lat_min:+6.1f}° to {lat_max:+6.1f}° (range: {lat_max - lat_min:4.1f}°)")
print(f"   Longitude: {lon_min:+6.1f}° to {lon_max:+6.1f}° (range: {lon_max - lon_min:4.1f}°)")

# Value mapping
point_values = {}
for i, point in enumerate(nagoya_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]  # Bottom-left corner
f_01 = point_values[(lat_min, lon_max)]  # Bottom-right corner
f_10 = point_values[(lat_max, lon_min)]  # Top-left corner
f_11 = point_values[(lat_max, lon_max)]  # Top-right corner

print(f"\n4. TEC VALUES MAPPING:")
print(f"   f_00 (lat_min, lon_min) = ({lat_min:+6.1f}°, {lon_min:+6.1f}°) = {f_00:5.1f}")
print(f"   f_01 (lat_min, lon_max) = ({lat_min:+6.1f}°, {lon_max:+6.1f}°) = {f_01:5.1f}")
print(f"   f_10 (lat_max, lon_min) = ({lat_max:+6.1f}°, {lon_min:+6.1f}°) = {f_10:5.1f}")
print(f"   f_11 (lat_max, lon_max) = ({lat_max:+6.1f}°, {lon_max:+6.1f}°) = {f_11:5.1f}")

# Interpolation parameters calculation
t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

print(f"\n5. INTERPOLATION PARAMETERS:")
print(f"   t_x = (target_lon - lon_min) / (lon_max - lon_min)")
print(f"   t_x = ({target_lon:+6.5f} - ({lon_min:+6.1f})) / ({lon_max:+6.1f} - ({lon_min:+6.1f}))")
print(f"   t_x = {target_lon - lon_min:6.5f} / {lon_max - lon_min:4.1f} = {t_x:.6f}")
print(f"   ")
print(f"   t_y = (target_lat - lat_min) / (lat_max - lat_min)")
print(f"   t_y = ({target_lat:+6.5f} - ({lat_min:+6.1f})) / ({lat_max:+6.1f} - ({lat_min:+6.1f}))")
print(f"   t_y = {target_lat - lat_min:6.5f} / {lat_max - lat_min:4.1f} = {t_y:.6f}")

# Weight calculations
w_00 = (1 - t_x) * (1 - t_y)
w_01 = t_x * (1 - t_y)
w_10 = (1 - t_x) * t_y
w_11 = t_x * t_y

print(f"\n6. INTERPOLATION WEIGHTS:")
print(f"   w_00 = (1 - t_x) * (1 - t_y) = (1 - {t_x:.6f}) * (1 - {t_y:.6f}) = {w_00:.6f}")
print(f"   w_01 = t_x * (1 - t_y)       = {t_x:.6f} * (1 - {t_y:.6f})       = {w_01:.6f}")
print(f"   w_10 = (1 - t_x) * t_y       = (1 - {t_x:.6f}) * {t_y:.6f}       = {w_10:.6f}")
print(f"   w_11 = t_x * t_y             = {t_x:.6f} * {t_y:.6f}             = {w_11:.6f}")
print(f"   Sum of weights = {w_00 + w_01 + w_10 + w_11:.6f} (should be 1.0)")

# Final interpolation
interpolated_value = w_00 * f_00 + w_01 * f_01 + w_10 * f_10 + w_11 * f_11

print(f"\n7. FINAL CALCULATION:")
print(f"   Value = w_00*f_00 + w_01*f_01 + w_10*f_10 + w_11*f_11")
print(f"   Value = {w_00:.6f}*{f_00:5.1f} + {w_01:.6f}*{f_01:5.1f} + {w_10:.6f}*{f_10:5.1f} + {w_11:.6f}*{f_11:5.1f}")
print(f"   Value = {w_00*f_00:8.4f} + {w_01*f_01:8.4f} + {w_10*f_10:8.4f} + {w_11*f_11:8.4f}")
print(f"   Value = {interpolated_value:.6f}")

print(f"\n8. TARGET POINT RELATIVE POSITION:")
print(f"   In X direction (longitude): {t_x*100:.2f}% of the way from {lon_min}° to {lon_max}°")
print(f"   In Y direction (latitude):  {t_y*100:.2f}% of the way from {lat_min}° to {lat_max}°")

print(f"\n{'='*60}")
print(f"FINAL RESULT: TEC = {interpolated_value:.4f}")
print(f"{'='*60}")

## 12/18/2024 - 18:30

In [ ]:
import pandas as pd

date = '2024-12-18'
time = '18:30:00'
timestamp_wanted = f'{date} {time}'

result = df_nagoya_maps_2024[df_nagoya_maps_2024['DATETIME'] == timestamp_wanted]

if result.empty:
    df_temp = df_nagoya_maps_2024.copy()
    df_temp['DATETIME'] = pd.to_datetime(df_temp['DATETIME'])
    timestamp_dt = pd.to_datetime(timestamp_wanted)

    df_temp['diff'] = abs((df_temp['DATETIME'] - timestamp_dt).dt.total_seconds())
    idx_nearest = df_temp['diff'].idxmin()

    result = df_nagoya_maps_2024.iloc[[idx_nearest]]
    print(f"Exact date not found. Using closest record: {df_nagoya_maps_2024.iloc[idx_nearest]['DATETIME']}")

In [ ]:
result

In [ ]:
nagoya_maps = np.array(result.iloc[:]['TECMAP'])

In [ ]:
np_nagoya_maps = []
for i in range(len(nagoya_maps)):
    np_nagoya_maps.append(nagoya_maps[i])
np_nagoya_maps = np.array(np_nagoya_maps)

In [ ]:
np_nagoya_maps

In [ ]:
np_nagoya_maps.shape

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80.4, 40.1),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class nagoya(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80.4, 40.1),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        super().__init__(extent, lat_step, lon_step)


nagoya_tec = nagoya(extent=(-110, 0, -80.4, 40.1), lat_step=0.5, lon_step=0.5)
nagoya_points = nagoya_tec.find_nearest_points(lat=-2.59346, lon=-44.21248)
print(nagoya_points)

In [ ]:
nagoya_points

In [ ]:
nagoya_tec.tec_map = np_nagoya_maps[0]

print("=" * 60)
print("DETAILED BILINEAR INTERPOLATION ANALYSIS")
print("=" * 60)

print("\n1. THE 4 CLOSEST POINTS:")
for i, point in enumerate(nagoya_points):
    lat, lon, lat_idx, lon_idx = point
    tec_value = nagoya_tec.tec_map[lat_idx, lon_idx]
    print(f"   P{i+1}: ({lat:+6.1f}°, {lon:+6.1f}°) - Indexes: [{lat_idx:2d}, {lon_idx:2d}] - TEC: {tec_value:5.1f}")

target_lat = -2.59346
target_lon = -44.21248

print(f"\n2. TARGET POINT: ({target_lat:+6.5f}°, {target_lon:+6.5f}°)")

# Collect values and coordinates
values = []
for point in nagoya_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(nagoya_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in nagoya_points]
lons = [p[1] for p in nagoya_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

print(f"\n3. INTERPOLATION RECTANGLE BOUNDS:")
print(f"   Latitude:  {lat_min:+6.1f}° to {lat_max:+6.1f}° (range: {lat_max - lat_min:4.1f}°)")
print(f"   Longitude: {lon_min:+6.1f}° to {lon_max:+6.1f}° (range: {lon_max - lon_min:4.1f}°)")

# Value mapping
point_values = {}
for i, point in enumerate(nagoya_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]  # Bottom-left corner
f_01 = point_values[(lat_min, lon_max)]  # Bottom-right corner
f_10 = point_values[(lat_max, lon_min)]  # Top-left corner
f_11 = point_values[(lat_max, lon_max)]  # Top-right corner

print(f"\n4. TEC VALUES MAPPING:")
print(f"   f_00 (lat_min, lon_min) = ({lat_min:+6.1f}°, {lon_min:+6.1f}°) = {f_00:5.1f}")
print(f"   f_01 (lat_min, lon_max) = ({lat_min:+6.1f}°, {lon_max:+6.1f}°) = {f_01:5.1f}")
print(f"   f_10 (lat_max, lon_min) = ({lat_max:+6.1f}°, {lon_min:+6.1f}°) = {f_10:5.1f}")
print(f"   f_11 (lat_max, lon_max) = ({lat_max:+6.1f}°, {lon_max:+6.1f}°) = {f_11:5.1f}")

# Interpolation parameters calculation
t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

print(f"\n5. INTERPOLATION PARAMETERS:")
print(f"   t_x = (target_lon - lon_min) / (lon_max - lon_min)")
print(f"   t_x = ({target_lon:+6.5f} - ({lon_min:+6.1f})) / ({lon_max:+6.1f} - ({lon_min:+6.1f}))")
print(f"   t_x = {target_lon - lon_min:6.5f} / {lon_max - lon_min:4.1f} = {t_x:.6f}")
print(f"   ")
print(f"   t_y = (target_lat - lat_min) / (lat_max - lat_min)")
print(f"   t_y = ({target_lat:+6.5f} - ({lat_min:+6.1f})) / ({lat_max:+6.1f} - ({lat_min:+6.1f}))")
print(f"   t_y = {target_lat - lat_min:6.5f} / {lat_max - lat_min:4.1f} = {t_y:.6f}")

# Weight calculations
w_00 = (1 - t_x) * (1 - t_y)
w_01 = t_x * (1 - t_y)
w_10 = (1 - t_x) * t_y
w_11 = t_x * t_y

print(f"\n6. INTERPOLATION WEIGHTS:")
print(f"   w_00 = (1 - t_x) * (1 - t_y) = (1 - {t_x:.6f}) * (1 - {t_y:.6f}) = {w_00:.6f}")
print(f"   w_01 = t_x * (1 - t_y)       = {t_x:.6f} * (1 - {t_y:.6f})       = {w_01:.6f}")
print(f"   w_10 = (1 - t_x) * t_y       = (1 - {t_x:.6f}) * {t_y:.6f}       = {w_10:.6f}")
print(f"   w_11 = t_x * t_y             = {t_x:.6f} * {t_y:.6f}             = {w_11:.6f}")
print(f"   Sum of weights = {w_00 + w_01 + w_10 + w_11:.6f} (should be 1.0)")

# Final interpolation
interpolated_value = w_00 * f_00 + w_01 * f_01 + w_10 * f_10 + w_11 * f_11

print(f"\n7. FINAL CALCULATION:")
print(f"   Value = w_00*f_00 + w_01*f_01 + w_10*f_10 + w_11*f_11")
print(f"   Value = {w_00:.6f}*{f_00:5.1f} + {w_01:.6f}*{f_01:5.1f} + {w_10:.6f}*{f_10:5.1f} + {w_11:.6f}*{f_11:5.1f}")
print(f"   Value = {w_00*f_00:8.4f} + {w_01*f_01:8.4f} + {w_10*f_10:8.4f} + {w_11*f_11:8.4f}")
print(f"   Value = {interpolated_value:.6f}")

print(f"\n8. TARGET POINT RELATIVE POSITION:")
print(f"   In X direction (longitude): {t_x*100:.2f}% of the way from {lon_min}° to {lon_max}°")
print(f"   In Y direction (latitude):  {t_y*100:.2f}% of the way from {lat_min}° to {lat_max}°")

print(f"\n{'='*60}")
print(f"FINAL RESULT: TEC = {interpolated_value:.4f}")
print(f"{'='*60}")

## 12/18/2024 - 18:00

In [ ]:
import pandas as pd

date = '2024-12-18'
time = '18:00:00'
timestamp_wanted = f'{date} {time}'

result = df_nagoya_maps_2024[df_nagoya_maps_2024['DATETIME'] == timestamp_wanted]

if result.empty:
    df_temp = df_nagoya_maps_2024.copy()
    df_temp['DATETIME'] = pd.to_datetime(df_temp['DATETIME'])
    timestamp_dt = pd.to_datetime(timestamp_wanted)

    df_temp['diff'] = abs((df_temp['DATETIME'] - timestamp_dt).dt.total_seconds())
    idx_nearest = df_temp['diff'].idxmin()

    result = df_nagoya_maps_2024.iloc[[idx_nearest]]
    print(f"Exact date not found. Using closest record: {df_nagoya_maps_2024.iloc[idx_nearest]['DATETIME']}")

In [ ]:
result

In [ ]:
nagoya_maps = np.array(result.iloc[:]['TECMAP'])

In [ ]:
np_nagoya_maps = []
for i in range(len(nagoya_maps)):
    np_nagoya_maps.append(nagoya_maps[i])
np_nagoya_maps = np.array(np_nagoya_maps)

In [ ]:
np_nagoya_maps

In [ ]:
np_nagoya_maps.shape

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80.4, 40.1),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class nagoya(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80.4, 40.1),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        super().__init__(extent, lat_step, lon_step)


nagoya_tec = nagoya(extent=(-110, 0, -80.4, 40.1), lat_step=0.5, lon_step=0.5)
nagoya_points = nagoya_tec.find_nearest_points(lat=-2.59346, lon=-44.21248)
print(nagoya_points)

In [ ]:
nagoya_points

In [ ]:
nagoya_tec.tec_map = np_nagoya_maps[0]

print("=" * 60)
print("DETAILED BILINEAR INTERPOLATION ANALYSIS")
print("=" * 60)

print("\n1. THE 4 CLOSEST POINTS:")
for i, point in enumerate(nagoya_points):
    lat, lon, lat_idx, lon_idx = point
    tec_value = nagoya_tec.tec_map[lat_idx, lon_idx]
    print(f"   P{i+1}: ({lat:+6.1f}°, {lon:+6.1f}°) - Indexes: [{lat_idx:2d}, {lon_idx:2d}] - TEC: {tec_value:5.1f}")

target_lat = -2.59346
target_lon = -44.21248

print(f"\n2. TARGET POINT: ({target_lat:+6.5f}°, {target_lon:+6.5f}°)")

# Collect values and coordinates
values = []
for point in nagoya_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(nagoya_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in nagoya_points]
lons = [p[1] for p in nagoya_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

print(f"\n3. INTERPOLATION RECTANGLE BOUNDS:")
print(f"   Latitude:  {lat_min:+6.1f}° to {lat_max:+6.1f}° (range: {lat_max - lat_min:4.1f}°)")
print(f"   Longitude: {lon_min:+6.1f}° to {lon_max:+6.1f}° (range: {lon_max - lon_min:4.1f}°)")

# Value mapping
point_values = {}
for i, point in enumerate(nagoya_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]  # Bottom-left corner
f_01 = point_values[(lat_min, lon_max)]  # Bottom-right corner
f_10 = point_values[(lat_max, lon_min)]  # Top-left corner
f_11 = point_values[(lat_max, lon_max)]  # Top-right corner

print(f"\n4. TEC VALUES MAPPING:")
print(f"   f_00 (lat_min, lon_min) = ({lat_min:+6.1f}°, {lon_min:+6.1f}°) = {f_00:5.1f}")
print(f"   f_01 (lat_min, lon_max) = ({lat_min:+6.1f}°, {lon_max:+6.1f}°) = {f_01:5.1f}")
print(f"   f_10 (lat_max, lon_min) = ({lat_max:+6.1f}°, {lon_min:+6.1f}°) = {f_10:5.1f}")
print(f"   f_11 (lat_max, lon_max) = ({lat_max:+6.1f}°, {lon_max:+6.1f}°) = {f_11:5.1f}")

# Interpolation parameters calculation
t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

print(f"\n5. INTERPOLATION PARAMETERS:")
print(f"   t_x = (target_lon - lon_min) / (lon_max - lon_min)")
print(f"   t_x = ({target_lon:+6.5f} - ({lon_min:+6.1f})) / ({lon_max:+6.1f} - ({lon_min:+6.1f}))")
print(f"   t_x = {target_lon - lon_min:6.5f} / {lon_max - lon_min:4.1f} = {t_x:.6f}")
print(f"   ")
print(f"   t_y = (target_lat - lat_min) / (lat_max - lat_min)")
print(f"   t_y = ({target_lat:+6.5f} - ({lat_min:+6.1f})) / ({lat_max:+6.1f} - ({lat_min:+6.1f}))")
print(f"   t_y = {target_lat - lat_min:6.5f} / {lat_max - lat_min:4.1f} = {t_y:.6f}")

# Weight calculations
w_00 = (1 - t_x) * (1 - t_y)
w_01 = t_x * (1 - t_y)
w_10 = (1 - t_x) * t_y
w_11 = t_x * t_y

print(f"\n6. INTERPOLATION WEIGHTS:")
print(f"   w_00 = (1 - t_x) * (1 - t_y) = (1 - {t_x:.6f}) * (1 - {t_y:.6f}) = {w_00:.6f}")
print(f"   w_01 = t_x * (1 - t_y)       = {t_x:.6f} * (1 - {t_y:.6f})       = {w_01:.6f}")
print(f"   w_10 = (1 - t_x) * t_y       = (1 - {t_x:.6f}) * {t_y:.6f}       = {w_10:.6f}")
print(f"   w_11 = t_x * t_y             = {t_x:.6f} * {t_y:.6f}             = {w_11:.6f}")
print(f"   Sum of weights = {w_00 + w_01 + w_10 + w_11:.6f} (should be 1.0)")

# Final interpolation
interpolated_value = w_00 * f_00 + w_01 * f_01 + w_10 * f_10 + w_11 * f_11

print(f"\n7. FINAL CALCULATION:")
print(f"   Value = w_00*f_00 + w_01*f_01 + w_10*f_10 + w_11*f_11")
print(f"   Value = {w_00:.6f}*{f_00:5.1f} + {w_01:.6f}*{f_01:5.1f} + {w_10:.6f}*{f_10:5.1f} + {w_11:.6f}*{f_11:5.1f}")
print(f"   Value = {w_00*f_00:8.4f} + {w_01*f_01:8.4f} + {w_10*f_10:8.4f} + {w_11*f_11:8.4f}")
print(f"   Value = {interpolated_value:.6f}")

print(f"\n8. TARGET POINT RELATIVE POSITION:")
print(f"   In X direction (longitude): {t_x*100:.2f}% of the way from {lon_min}° to {lon_max}°")
print(f"   In Y direction (latitude):  {t_y*100:.2f}% of the way from {lat_min}° to {lat_max}°")

print(f"\n{'='*60}")
print(f"FINAL RESULT: TEC = {interpolated_value:.4f}")
print(f"{'='*60}")

## 12/22/2024 - 18:00

In [ ]:
import pandas as pd

date = '2024-12-22'
time = '18:00:00'
timestamp_wanted = f'{date} {time}'

result = df_nagoya_maps_2024[df_nagoya_maps_2024['DATETIME'] == timestamp_wanted]

if result.empty:
    df_temp = df_nagoya_maps_2024.copy()
    df_temp['DATETIME'] = pd.to_datetime(df_temp['DATETIME'])
    timestamp_dt = pd.to_datetime(timestamp_wanted)

    df_temp['diff'] = abs((df_temp['DATETIME'] - timestamp_dt).dt.total_seconds())
    idx_nearest = df_temp['diff'].idxmin()

    result = df_nagoya_maps_2024.iloc[[idx_nearest]]
    print(f"Exact date not found. Using closest record: {df_nagoya_maps_2024.iloc[idx_nearest]['DATETIME']}")

In [ ]:
result

In [ ]:
nagoya_maps = np.array(result.iloc[:]['TECMAP'])

In [ ]:
np_nagoya_maps = []
for i in range(len(nagoya_maps)):
    np_nagoya_maps.append(nagoya_maps[i])
np_nagoya_maps = np.array(np_nagoya_maps)

In [ ]:
np_nagoya_maps

In [ ]:
np_nagoya_maps.shape

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80.4, 40.1),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class nagoya(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80.4, 40.1),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        super().__init__(extent, lat_step, lon_step)


nagoya_tec = nagoya(extent=(-110, 0, -80.4, 40.1), lat_step=0.5, lon_step=0.5)
nagoya_points = nagoya_tec.find_nearest_points(lat=-2.59346, lon=-44.21248)
print(nagoya_points)

In [ ]:
nagoya_points

In [ ]:
nagoya_tec.tec_map = np_nagoya_maps[0]

print("=" * 60)
print("DETAILED BILINEAR INTERPOLATION ANALYSIS")
print("=" * 60)

print("\n1. THE 4 CLOSEST POINTS:")
for i, point in enumerate(nagoya_points):
    lat, lon, lat_idx, lon_idx = point
    tec_value = nagoya_tec.tec_map[lat_idx, lon_idx]
    print(f"   P{i+1}: ({lat:+6.1f}°, {lon:+6.1f}°) - Indexes: [{lat_idx:2d}, {lon_idx:2d}] - TEC: {tec_value:5.1f}")

target_lat = -2.59346
target_lon = -44.21248

print(f"\n2. TARGET POINT: ({target_lat:+6.5f}°, {target_lon:+6.5f}°)")

# Collect values and coordinates
values = []
for point in nagoya_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(nagoya_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in nagoya_points]
lons = [p[1] for p in nagoya_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

print(f"\n3. INTERPOLATION RECTANGLE BOUNDS:")
print(f"   Latitude:  {lat_min:+6.1f}° to {lat_max:+6.1f}° (range: {lat_max - lat_min:4.1f}°)")
print(f"   Longitude: {lon_min:+6.1f}° to {lon_max:+6.1f}° (range: {lon_max - lon_min:4.1f}°)")

# Value mapping
point_values = {}
for i, point in enumerate(nagoya_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]  # Bottom-left corner
f_01 = point_values[(lat_min, lon_max)]  # Bottom-right corner
f_10 = point_values[(lat_max, lon_min)]  # Top-left corner
f_11 = point_values[(lat_max, lon_max)]  # Top-right corner

print(f"\n4. TEC VALUES MAPPING:")
print(f"   f_00 (lat_min, lon_min) = ({lat_min:+6.1f}°, {lon_min:+6.1f}°) = {f_00:5.1f}")
print(f"   f_01 (lat_min, lon_max) = ({lat_min:+6.1f}°, {lon_max:+6.1f}°) = {f_01:5.1f}")
print(f"   f_10 (lat_max, lon_min) = ({lat_max:+6.1f}°, {lon_min:+6.1f}°) = {f_10:5.1f}")
print(f"   f_11 (lat_max, lon_max) = ({lat_max:+6.1f}°, {lon_max:+6.1f}°) = {f_11:5.1f}")

# Interpolation parameters calculation
t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

print(f"\n5. INTERPOLATION PARAMETERS:")
print(f"   t_x = (target_lon - lon_min) / (lon_max - lon_min)")
print(f"   t_x = ({target_lon:+6.5f} - ({lon_min:+6.1f})) / ({lon_max:+6.1f} - ({lon_min:+6.1f}))")
print(f"   t_x = {target_lon - lon_min:6.5f} / {lon_max - lon_min:4.1f} = {t_x:.6f}")
print(f"   ")
print(f"   t_y = (target_lat - lat_min) / (lat_max - lat_min)")
print(f"   t_y = ({target_lat:+6.5f} - ({lat_min:+6.1f})) / ({lat_max:+6.1f} - ({lat_min:+6.1f}))")
print(f"   t_y = {target_lat - lat_min:6.5f} / {lat_max - lat_min:4.1f} = {t_y:.6f}")

# Weight calculations
w_00 = (1 - t_x) * (1 - t_y)
w_01 = t_x * (1 - t_y)
w_10 = (1 - t_x) * t_y
w_11 = t_x * t_y

print(f"\n6. INTERPOLATION WEIGHTS:")
print(f"   w_00 = (1 - t_x) * (1 - t_y) = (1 - {t_x:.6f}) * (1 - {t_y:.6f}) = {w_00:.6f}")
print(f"   w_01 = t_x * (1 - t_y)       = {t_x:.6f} * (1 - {t_y:.6f})       = {w_01:.6f}")
print(f"   w_10 = (1 - t_x) * t_y       = (1 - {t_x:.6f}) * {t_y:.6f}       = {w_10:.6f}")
print(f"   w_11 = t_x * t_y             = {t_x:.6f} * {t_y:.6f}             = {w_11:.6f}")
print(f"   Sum of weights = {w_00 + w_01 + w_10 + w_11:.6f} (should be 1.0)")

# Final interpolation
interpolated_value = w_00 * f_00 + w_01 * f_01 + w_10 * f_10 + w_11 * f_11

print(f"\n7. FINAL CALCULATION:")
print(f"   Value = w_00*f_00 + w_01*f_01 + w_10*f_10 + w_11*f_11")
print(f"   Value = {w_00:.6f}*{f_00:5.1f} + {w_01:.6f}*{f_01:5.1f} + {w_10:.6f}*{f_10:5.1f} + {w_11:.6f}*{f_11:5.1f}")
print(f"   Value = {w_00*f_00:8.4f} + {w_01*f_01:8.4f} + {w_10*f_10:8.4f} + {w_11*f_11:8.4f}")
print(f"   Value = {interpolated_value:.6f}")

print(f"\n8. TARGET POINT RELATIVE POSITION:")
print(f"   In X direction (longitude): {t_x*100:.2f}% of the way from {lon_min}° to {lon_max}°")
print(f"   In Y direction (latitude):  {t_y*100:.2f}% of the way from {lat_min}° to {lat_max}°")

print(f"\n{'='*60}")
print(f"FINAL RESULT: TEC = {interpolated_value:.4f}")
print(f"{'='*60}")

## 12/19/2024 - 18:00

In [ ]:
import pandas as pd

date = '2024-12-19'
time = '18:00:00'
timestamp_wanted = f'{date} {time}'

result = df_nagoya_maps_2024[df_nagoya_maps_2024['DATETIME'] == timestamp_wanted]

if result.empty:
    df_temp = df_nagoya_maps_2024.copy()
    df_temp['DATETIME'] = pd.to_datetime(df_temp['DATETIME'])
    timestamp_dt = pd.to_datetime(timestamp_wanted)

    df_temp['diff'] = abs((df_temp['DATETIME'] - timestamp_dt).dt.total_seconds())
    idx_nearest = df_temp['diff'].idxmin()

    result = df_nagoya_maps_2024.iloc[[idx_nearest]]
    print(f"Exact date not found. Using closest record: {df_nagoya_maps_2024.iloc[idx_nearest]['DATETIME']}")

In [ ]:
result

In [ ]:
nagoya_maps = np.array(result.iloc[:]['TECMAP'])

In [ ]:
np_nagoya_maps = []
for i in range(len(nagoya_maps)):
    np_nagoya_maps.append(nagoya_maps[i])
np_nagoya_maps = np.array(np_nagoya_maps)

In [ ]:
np_nagoya_maps

In [ ]:
np_nagoya_maps.shape

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80.4, 40.1),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class nagoya(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80.4, 40.1),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        super().__init__(extent, lat_step, lon_step)


nagoya_tec = nagoya(extent=(-110, 0, -80.4, 40.1), lat_step=0.5, lon_step=0.5)
nagoya_points = nagoya_tec.find_nearest_points(lat=-2.59346, lon=-44.21248)
print(nagoya_points)

In [ ]:
nagoya_points

In [ ]:
nagoya_tec.tec_map = np_nagoya_maps[0]

print("=" * 60)
print("DETAILED BILINEAR INTERPOLATION ANALYSIS")
print("=" * 60)

print("\n1. THE 4 CLOSEST POINTS:")
for i, point in enumerate(nagoya_points):
    lat, lon, lat_idx, lon_idx = point
    tec_value = nagoya_tec.tec_map[lat_idx, lon_idx]
    print(f"   P{i+1}: ({lat:+6.1f}°, {lon:+6.1f}°) - Indexes: [{lat_idx:2d}, {lon_idx:2d}] - TEC: {tec_value:5.1f}")

target_lat = -2.59346
target_lon = -44.21248

print(f"\n2. TARGET POINT: ({target_lat:+6.5f}°, {target_lon:+6.5f}°)")

# Collect values and coordinates
values = []
for point in nagoya_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(nagoya_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in nagoya_points]
lons = [p[1] for p in nagoya_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

print(f"\n3. INTERPOLATION RECTANGLE BOUNDS:")
print(f"   Latitude:  {lat_min:+6.1f}° to {lat_max:+6.1f}° (range: {lat_max - lat_min:4.1f}°)")
print(f"   Longitude: {lon_min:+6.1f}° to {lon_max:+6.1f}° (range: {lon_max - lon_min:4.1f}°)")

# Value mapping
point_values = {}
for i, point in enumerate(nagoya_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]  # Bottom-left corner
f_01 = point_values[(lat_min, lon_max)]  # Bottom-right corner
f_10 = point_values[(lat_max, lon_min)]  # Top-left corner
f_11 = point_values[(lat_max, lon_max)]  # Top-right corner

print(f"\n4. TEC VALUES MAPPING:")
print(f"   f_00 (lat_min, lon_min) = ({lat_min:+6.1f}°, {lon_min:+6.1f}°) = {f_00:5.1f}")
print(f"   f_01 (lat_min, lon_max) = ({lat_min:+6.1f}°, {lon_max:+6.1f}°) = {f_01:5.1f}")
print(f"   f_10 (lat_max, lon_min) = ({lat_max:+6.1f}°, {lon_min:+6.1f}°) = {f_10:5.1f}")
print(f"   f_11 (lat_max, lon_max) = ({lat_max:+6.1f}°, {lon_max:+6.1f}°) = {f_11:5.1f}")

# Interpolation parameters calculation
t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

print(f"\n5. INTERPOLATION PARAMETERS:")
print(f"   t_x = (target_lon - lon_min) / (lon_max - lon_min)")
print(f"   t_x = ({target_lon:+6.5f} - ({lon_min:+6.1f})) / ({lon_max:+6.1f} - ({lon_min:+6.1f}))")
print(f"   t_x = {target_lon - lon_min:6.5f} / {lon_max - lon_min:4.1f} = {t_x:.6f}")
print(f"   ")
print(f"   t_y = (target_lat - lat_min) / (lat_max - lat_min)")
print(f"   t_y = ({target_lat:+6.5f} - ({lat_min:+6.1f})) / ({lat_max:+6.1f} - ({lat_min:+6.1f}))")
print(f"   t_y = {target_lat - lat_min:6.5f} / {lat_max - lat_min:4.1f} = {t_y:.6f}")

# Weight calculations
w_00 = (1 - t_x) * (1 - t_y)
w_01 = t_x * (1 - t_y)
w_10 = (1 - t_x) * t_y
w_11 = t_x * t_y

print(f"\n6. INTERPOLATION WEIGHTS:")
print(f"   w_00 = (1 - t_x) * (1 - t_y) = (1 - {t_x:.6f}) * (1 - {t_y:.6f}) = {w_00:.6f}")
print(f"   w_01 = t_x * (1 - t_y)       = {t_x:.6f} * (1 - {t_y:.6f})       = {w_01:.6f}")
print(f"   w_10 = (1 - t_x) * t_y       = (1 - {t_x:.6f}) * {t_y:.6f}       = {w_10:.6f}")
print(f"   w_11 = t_x * t_y             = {t_x:.6f} * {t_y:.6f}             = {w_11:.6f}")
print(f"   Sum of weights = {w_00 + w_01 + w_10 + w_11:.6f} (should be 1.0)")

# Final interpolation
interpolated_value = w_00 * f_00 + w_01 * f_01 + w_10 * f_10 + w_11 * f_11

print(f"\n7. FINAL CALCULATION:")
print(f"   Value = w_00*f_00 + w_01*f_01 + w_10*f_10 + w_11*f_11")
print(f"   Value = {w_00:.6f}*{f_00:5.1f} + {w_01:.6f}*{f_01:5.1f} + {w_10:.6f}*{f_10:5.1f} + {w_11:.6f}*{f_11:5.1f}")
print(f"   Value = {w_00*f_00:8.4f} + {w_01*f_01:8.4f} + {w_10*f_10:8.4f} + {w_11*f_11:8.4f}")
print(f"   Value = {interpolated_value:.6f}")

print(f"\n8. TARGET POINT RELATIVE POSITION:")
print(f"   In X direction (longitude): {t_x*100:.2f}% of the way from {lon_min}° to {lon_max}°")
print(f"   In Y direction (latitude):  {t_y*100:.2f}% of the way from {lat_min}° to {lat_max}°")

print(f"\n{'='*60}")
print(f"FINAL RESULT: TEC = {interpolated_value:.4f}")
print(f"{'='*60}")

## 12/01/2024 - 18:00

In [ ]:
import pandas as pd

date = '2024-12-01'
time = '18:00:00'
timestamp_wanted = f'{date} {time}'

result = df_nagoya_maps_2024[df_nagoya_maps_2024['DATETIME'] == timestamp_wanted]

if result.empty:
    df_temp = df_nagoya_maps_2024.copy()
    df_temp['DATETIME'] = pd.to_datetime(df_temp['DATETIME'])
    timestamp_dt = pd.to_datetime(timestamp_wanted)

    df_temp['diff'] = abs((df_temp['DATETIME'] - timestamp_dt).dt.total_seconds())
    idx_nearest = df_temp['diff'].idxmin()

    result = df_nagoya_maps_2024.iloc[[idx_nearest]]
    print(f"Exact date not found. Using closest record: {df_nagoya_maps_2024.iloc[idx_nearest]['DATETIME']}")

In [ ]:
result

In [ ]:
nagoya_maps = np.array(result.iloc[:]['TECMAP'])

In [ ]:
np_nagoya_maps = []
for i in range(len(nagoya_maps)):
    np_nagoya_maps.append(nagoya_maps[i])
np_nagoya_maps = np.array(np_nagoya_maps)

In [ ]:
np_nagoya_maps

In [ ]:
np_nagoya_maps.shape

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80.4, 40.1),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class nagoya(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80.4, 40.1),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        super().__init__(extent, lat_step, lon_step)


nagoya_tec = nagoya(extent=(-110, 0, -80.4, 40.1), lat_step=0.5, lon_step=0.5)
nagoya_points = nagoya_tec.find_nearest_points(lat=-2.59346, lon=-45.51)
print(nagoya_points)

In [ ]:
nagoya_points

In [ ]:
nagoya_tec.tec_map = np_nagoya_maps[0]

print("=" * 60)
print("DETAILED BILINEAR INTERPOLATION ANALYSIS")
print("=" * 60)

print("\n1. THE 4 CLOSEST POINTS:")
for i, point in enumerate(nagoya_points):
    lat, lon, lat_idx, lon_idx = point
    tec_value = nagoya_tec.tec_map[lat_idx, lon_idx]
    print(f"   P{i+1}: ({lat:+6.1f}°, {lon:+6.1f}°) - Indexes: [{lat_idx:2d}, {lon_idx:2d}] - TEC: {tec_value:5.1f}")

target_lat = -2.59346
target_lon = -44.21248

print(f"\n2. TARGET POINT: ({target_lat:+6.5f}°, {target_lon:+6.5f}°)")

# Collect values and coordinates
values = []
for point in nagoya_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(nagoya_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in nagoya_points]
lons = [p[1] for p in nagoya_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

print(f"\n3. INTERPOLATION RECTANGLE BOUNDS:")
print(f"   Latitude:  {lat_min:+6.1f}° to {lat_max:+6.1f}° (range: {lat_max - lat_min:4.1f}°)")
print(f"   Longitude: {lon_min:+6.1f}° to {lon_max:+6.1f}° (range: {lon_max - lon_min:4.1f}°)")

# Value mapping
point_values = {}
for i, point in enumerate(nagoya_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]  # Bottom-left corner
f_01 = point_values[(lat_min, lon_max)]  # Bottom-right corner
f_10 = point_values[(lat_max, lon_min)]  # Top-left corner
f_11 = point_values[(lat_max, lon_max)]  # Top-right corner

print(f"\n4. TEC VALUES MAPPING:")
print(f"   f_00 (lat_min, lon_min) = ({lat_min:+6.1f}°, {lon_min:+6.1f}°) = {f_00:5.1f}")
print(f"   f_01 (lat_min, lon_max) = ({lat_min:+6.1f}°, {lon_max:+6.1f}°) = {f_01:5.1f}")
print(f"   f_10 (lat_max, lon_min) = ({lat_max:+6.1f}°, {lon_min:+6.1f}°) = {f_10:5.1f}")
print(f"   f_11 (lat_max, lon_max) = ({lat_max:+6.1f}°, {lon_max:+6.1f}°) = {f_11:5.1f}")

# Interpolation parameters calculation
t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

print(f"\n5. INTERPOLATION PARAMETERS:")
print(f"   t_x = (target_lon - lon_min) / (lon_max - lon_min)")
print(f"   t_x = ({target_lon:+6.5f} - ({lon_min:+6.1f})) / ({lon_max:+6.1f} - ({lon_min:+6.1f}))")
print(f"   t_x = {target_lon - lon_min:6.5f} / {lon_max - lon_min:4.1f} = {t_x:.6f}")
print(f"   ")
print(f"   t_y = (target_lat - lat_min) / (lat_max - lat_min)")
print(f"   t_y = ({target_lat:+6.5f} - ({lat_min:+6.1f})) / ({lat_max:+6.1f} - ({lat_min:+6.1f}))")
print(f"   t_y = {target_lat - lat_min:6.5f} / {lat_max - lat_min:4.1f} = {t_y:.6f}")

# Weight calculations
w_00 = (1 - t_x) * (1 - t_y)
w_01 = t_x * (1 - t_y)
w_10 = (1 - t_x) * t_y
w_11 = t_x * t_y

print(f"\n6. INTERPOLATION WEIGHTS:")
print(f"   w_00 = (1 - t_x) * (1 - t_y) = (1 - {t_x:.6f}) * (1 - {t_y:.6f}) = {w_00:.6f}")
print(f"   w_01 = t_x * (1 - t_y)       = {t_x:.6f} * (1 - {t_y:.6f})       = {w_01:.6f}")
print(f"   w_10 = (1 - t_x) * t_y       = (1 - {t_x:.6f}) * {t_y:.6f}       = {w_10:.6f}")
print(f"   w_11 = t_x * t_y             = {t_x:.6f} * {t_y:.6f}             = {w_11:.6f}")
print(f"   Sum of weights = {w_00 + w_01 + w_10 + w_11:.6f} (should be 1.0)")

# Final interpolation
interpolated_value = w_00 * f_00 + w_01 * f_01 + w_10 * f_10 + w_11 * f_11

print(f"\n7. FINAL CALCULATION:")
print(f"   Value = w_00*f_00 + w_01*f_01 + w_10*f_10 + w_11*f_11")
print(f"   Value = {w_00:.6f}*{f_00:5.1f} + {w_01:.6f}*{f_01:5.1f} + {w_10:.6f}*{f_10:5.1f} + {w_11:.6f}*{f_11:5.1f}")
print(f"   Value = {w_00*f_00:8.4f} + {w_01*f_01:8.4f} + {w_10*f_10:8.4f} + {w_11*f_11:8.4f}")
print(f"   Value = {interpolated_value:.6f}")

print(f"\n8. TARGET POINT RELATIVE POSITION:")
print(f"   In X direction (longitude): {t_x*100:.2f}% of the way from {lon_min}° to {lon_max}°")
print(f"   In Y direction (latitude):  {t_y*100:.2f}% of the way from {lat_min}° to {lat_max}°")

print(f"\n{'='*60}")
print(f"FINAL RESULT: TEC = {interpolated_value:.4f}")
print(f"{'='*60}")

# EMBRACE

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
#Download TF_EMBRACE_TEC_maps_intersection_case_study_from_Sept_05_2024_to_Dec_2024.pkl from https://doi.org/10.5281/zenodo.15453941

In [ ]:
!gdown '1LkY0IeFfdnahoH81OKYW-2xmXHgjFPhH'

In [ ]:
df_embrace_maps_2024 = pd.read_pickle('/content/TF_EMBRACE_TEC_maps_intersection_case_study_from_Sept_05_2024_to_Dec_2024.pkl')

In [ ]:
df_embrace_maps_2024

## 12/18/2024 - 18:20

In [ ]:
import pandas as pd

date = '2024-12-18'
time = '18:20:00'
timestamp_wanted = f'{date} {time}'

if df_embrace_maps_2024.index.name == 'DATETIME':
    result = df_embrace_maps_2024.loc[timestamp_wanted:timestamp_wanted]
else:
    if 'DATETIME' in df_embrace_maps_2024.columns:
        df_embrace_maps_2024 = df_embrace_maps_2024.set_index('DATETIME')
        result = df_embrace_maps_2024.loc[timestamp_wanted:timestamp_wanted]
    else:
        print("DATETIME column not found!")
        print("Available columns:", df_embrace_maps_2024.columns.tolist())

if result.empty:
    if not isinstance(df_embrace_maps_2024.index, pd.DatetimeIndex):
        df_embrace_maps_2024.index = pd.to_datetime(df_embrace_maps_2024.index)

    timestamp_dt = pd.to_datetime(timestamp_wanted)

    diference = abs((df_embrace_maps_2024.index - timestamp_dt).total_seconds())
    idx_nearest = diference.argmin()

    result = df_embrace_maps_2024.iloc[[idx_nearest]]
    print(f"Exact date not found. Using closest record: {df_embrace_maps_2024.index[idx_nearest]}")

In [ ]:
result

In [ ]:
embrace_maps = np.array(result.iloc[:]['TECMAP'])

In [ ]:
np_embrace_maps = []
for i in range(len(embrace_maps)):
    np_embrace_maps.append(embrace_maps[i])
np_embrace_maps = np.array(np_embrace_maps)

In [ ]:
np_embrace_maps

In [ ]:
np_embrace_maps.shape

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-90, -30, -60, 20),
                 lat_step: float = 2,
                 lon_step: float = 2.5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class embrace(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-90, -30, -60, 20),
                 lat_step: float = 2,
                 lon_step: float = 2.5):
        super().__init__(extent, lat_step, lon_step)


embrace_tec = embrace(extent=(-90, -30, -60, 20), lat_step=2, lon_step=2.5)
embrace_points = embrace_tec.find_nearest_points(lat=-2.59346, lon=-44.21248)
print(embrace_points)

In [ ]:
embrace_points

In [ ]:
embrace_tec.tec_map = np_embrace_maps[0]

print("=" * 60)
print("DETAILED BILINEAR INTERPOLATION ANALYSIS")
print("=" * 60)

print("\n1. THE 4 CLOSEST POINTS:")
for i, point in enumerate(embrace_points):
    lat, lon, lat_idx, lon_idx = point
    tec_value = embrace_tec.tec_map[lat_idx, lon_idx]
    print(f"   P{i+1}: ({lat:+6.1f}°, {lon:+6.1f}°) - Indexes: [{lat_idx:2d}, {lon_idx:2d}] - TEC: {tec_value:5.1f}")

target_lat = -2.59346
target_lon = -44.21248

print(f"\n2. TARGET POINT: ({target_lat:+6.5f}°, {target_lon:+6.5f}°)")

# Collect values and coordinates
values = []
for point in embrace_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(embrace_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in embrace_points]
lons = [p[1] for p in embrace_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

print(f"\n3. INTERPOLATION RECTANGLE BOUNDS:")
print(f"   Latitude:  {lat_min:+6.1f}° to {lat_max:+6.1f}° (range: {lat_max - lat_min:4.1f}°)")
print(f"   Longitude: {lon_min:+6.1f}° to {lon_max:+6.1f}° (range: {lon_max - lon_min:4.1f}°)")

# Value mapping
point_values = {}
for i, point in enumerate(embrace_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]  # Bottom-left corner
f_01 = point_values[(lat_min, lon_max)]  # Bottom-right corner
f_10 = point_values[(lat_max, lon_min)]  # Top-left corner
f_11 = point_values[(lat_max, lon_max)]  # Top-right corner

print(f"\n4. TEC VALUES MAPPING:")
print(f"   f_00 (lat_min, lon_min) = ({lat_min:+6.1f}°, {lon_min:+6.1f}°) = {f_00:5.1f}")
print(f"   f_01 (lat_min, lon_max) = ({lat_min:+6.1f}°, {lon_max:+6.1f}°) = {f_01:5.1f}")
print(f"   f_10 (lat_max, lon_min) = ({lat_max:+6.1f}°, {lon_min:+6.1f}°) = {f_10:5.1f}")
print(f"   f_11 (lat_max, lon_max) = ({lat_max:+6.1f}°, {lon_max:+6.1f}°) = {f_11:5.1f}")

# Interpolation parameters calculation
t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

print(f"\n5. INTERPOLATION PARAMETERS:")
print(f"   t_x = (target_lon - lon_min) / (lon_max - lon_min)")
print(f"   t_x = ({target_lon:+6.5f} - ({lon_min:+6.1f})) / ({lon_max:+6.1f} - ({lon_min:+6.1f}))")
print(f"   t_x = {target_lon - lon_min:6.5f} / {lon_max - lon_min:4.1f} = {t_x:.6f}")
print(f"   ")
print(f"   t_y = (target_lat - lat_min) / (lat_max - lat_min)")
print(f"   t_y = ({target_lat:+6.5f} - ({lat_min:+6.1f})) / ({lat_max:+6.1f} - ({lat_min:+6.1f}))")
print(f"   t_y = {target_lat - lat_min:6.5f} / {lat_max - lat_min:4.1f} = {t_y:.6f}")

# Weight calculations
w_00 = (1 - t_x) * (1 - t_y)
w_01 = t_x * (1 - t_y)
w_10 = (1 - t_x) * t_y
w_11 = t_x * t_y

print(f"\n6. INTERPOLATION WEIGHTS:")
print(f"   w_00 = (1 - t_x) * (1 - t_y) = (1 - {t_x:.6f}) * (1 - {t_y:.6f}) = {w_00:.6f}")
print(f"   w_01 = t_x * (1 - t_y)       = {t_x:.6f} * (1 - {t_y:.6f})       = {w_01:.6f}")
print(f"   w_10 = (1 - t_x) * t_y       = (1 - {t_x:.6f}) * {t_y:.6f}       = {w_10:.6f}")
print(f"   w_11 = t_x * t_y             = {t_x:.6f} * {t_y:.6f}             = {w_11:.6f}")
print(f"   Sum of weights = {w_00 + w_01 + w_10 + w_11:.6f} (should be 1.0)")

# Final interpolation
interpolated_value = w_00 * f_00 + w_01 * f_01 + w_10 * f_10 + w_11 * f_11

print(f"\n7. FINAL CALCULATION:")
print(f"   Value = w_00*f_00 + w_01*f_01 + w_10*f_10 + w_11*f_11")
print(f"   Value = {w_00:.6f}*{f_00:5.1f} + {w_01:.6f}*{f_01:5.1f} + {w_10:.6f}*{f_10:5.1f} + {w_11:.6f}*{f_11:5.1f}")
print(f"   Value = {w_00*f_00:8.4f} + {w_01*f_01:8.4f} + {w_10*f_10:8.4f} + {w_11*f_11:8.4f}")
print(f"   Value = {interpolated_value:.6f}")

print(f"\n8. TARGET POINT RELATIVE POSITION:")
print(f"   In X direction (longitude): {t_x*100:.2f}% of the way from {lon_min}° to {lon_max}°")
print(f"   In Y direction (latitude):  {t_y*100:.2f}% of the way from {lat_min}° to {lat_max}°")

print(f"\n{'='*60}")
print(f"FINAL RESULT: TEC = {interpolated_value:.4f}")
print(f"{'='*60}")

## 12/18/2024 - 18:10

In [ ]:
import pandas as pd

date = '2024-12-18'
time = '18:10:00'
timestamp_wanted = f'{date} {time}'

if df_embrace_maps_2024.index.name == 'DATETIME':
    result = df_embrace_maps_2024.loc[timestamp_wanted:timestamp_wanted]
else:
    if 'DATETIME' in df_embrace_maps_2024.columns:
        df_embrace_maps_2024 = df_embrace_maps_2024.set_index('DATETIME')
        result = df_embrace_maps_2024.loc[timestamp_wanted:timestamp_wanted]
    else:
        print("DATETIME column not found!")
        print("Available columns:", df_embrace_maps_2024.columns.tolist())

if result.empty:
    if not isinstance(df_embrace_maps_2024.index, pd.DatetimeIndex):
        df_embrace_maps_2024.index = pd.to_datetime(df_embrace_maps_2024.index)

    timestamp_dt = pd.to_datetime(timestamp_wanted)

    diference = abs((df_embrace_maps_2024.index - timestamp_dt).total_seconds())
    idx_nearest = diference.argmin()

    result = df_embrace_maps_2024.iloc[[idx_nearest]]
    print(f"Exact date not found. Using closest record: {df_embrace_maps_2024.index[idx_nearest]}")

In [ ]:
result

In [ ]:
embrace_maps = np.array(result.iloc[:]['TECMAP'])

In [ ]:
np_embrace_maps = []
for i in range(len(embrace_maps)):
    np_embrace_maps.append(embrace_maps[i])
np_embrace_maps = np.array(np_embrace_maps)

In [ ]:
np_embrace_maps

In [ ]:
np_embrace_maps.shape

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-90, -30, -60, 20),
                 lat_step: float = 2,
                 lon_step: float = 2.5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class embrace(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-90, -30, -60, 20),
                 lat_step: float = 2,
                 lon_step: float = 2.5):
        super().__init__(extent, lat_step, lon_step)


embrace_tec = embrace(extent=(-90, -30, -60, 20), lat_step=2, lon_step=2.5)
embrace_points = embrace_tec.find_nearest_points(lat=-2.59346, lon=-44.21248)
print(embrace_points)

In [ ]:
embrace_points

In [ ]:
embrace_tec.tec_map = np_embrace_maps[0]

print("=" * 60)
print("DETAILED BILINEAR INTERPOLATION ANALYSIS - EMBRACE")
print("=" * 60)

print("\n1. THE 4 CLOSEST POINTS:")
for i, point in enumerate(embrace_points):
    lat, lon, lat_idx, lon_idx = point
    tec_value = embrace_tec.tec_map[lat_idx, lon_idx]
    print(f"   P{i+1}: ({lat:+6.1f}°, {lon:+6.1f}°) - Indexes: [{lat_idx:2d}, {lon_idx:2d}] - TEC: {tec_value:5.1f}")

target_lat = -2.59346
target_lon = -44.21248

print(f"\n2. TARGET POINT: ({target_lat:+6.2f}°, {target_lon:+6.2f}°)")

# Collect values and coordinates
values = []
for point in embrace_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(embrace_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in embrace_points]
lons = [p[1] for p in embrace_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

print(f"\n3. INTERPOLATION RECTANGLE BOUNDS:")
print(f"   Latitude:  {lat_min:+6.1f}° to {lat_max:+6.1f}° (range: {lat_max - lat_min:4.1f}°)")
print(f"   Longitude: {lon_min:+6.1f}° to {lon_max:+6.1f}° (range: {lon_max - lon_min:4.1f}°)")

# Value mapping
point_values = {}
for i, point in enumerate(embrace_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]  # Bottom-left corner
f_01 = point_values[(lat_min, lon_max)]  # Bottom-right corner
f_10 = point_values[(lat_max, lon_min)]  # Top-left corner
f_11 = point_values[(lat_max, lon_max)]  # Top-right corner

print(f"\n4. TEC VALUES MAPPING:")
print(f"   f_00 (lat_min, lon_min) = ({lat_min:+6.1f}°, {lon_min:+6.1f}°) = {f_00:5.1f}")
print(f"   f_01 (lat_min, lon_max) = ({lat_min:+6.1f}°, {lon_max:+6.1f}°) = {f_01:5.1f}")
print(f"   f_10 (lat_max, lon_min) = ({lat_max:+6.1f}°, {lon_min:+6.1f}°) = {f_10:5.1f}")
print(f"   f_11 (lat_max, lon_max) = ({lat_max:+6.1f}°, {lon_max:+6.1f}°) = {f_11:5.1f}")

# Interpolation parameters calculation
t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

print(f"\n5. INTERPOLATION PARAMETERS:")
print(f"   t_x = (target_lon - lon_min) / (lon_max - lon_min)")
print(f"   t_x = ({target_lon:+6.2f} - ({lon_min:+6.1f})) / ({lon_max:+6.1f} - ({lon_min:+6.1f}))")
print(f"   t_x = {target_lon - lon_min:6.2f} / {lon_max - lon_min:4.1f} = {t_x:.6f}")
print(f"   ")
print(f"   t_y = (target_lat - lat_min) / (lat_max - lat_min)")
print(f"   t_y = ({target_lat:+6.2f} - ({lat_min:+6.1f})) / ({lat_max:+6.1f} - ({lat_min:+6.1f}))")
print(f"   t_y = {target_lat - lat_min:6.2f} / {lat_max - lat_min:4.1f} = {t_y:.6f}")

# Weight calculations
w_00 = (1 - t_x) * (1 - t_y)
w_01 = t_x * (1 - t_y)
w_10 = (1 - t_x) * t_y
w_11 = t_x * t_y

print(f"\n6. INTERPOLATION WEIGHTS:")
print(f"   w_00 = (1 - t_x) * (1 - t_y) = (1 - {t_x:.6f}) * (1 - {t_y:.6f}) = {w_00:.6f}")
print(f"   w_01 = t_x * (1 - t_y)       = {t_x:.6f} * (1 - {t_y:.6f})       = {w_01:.6f}")
print(f"   w_10 = (1 - t_x) * t_y       = (1 - {t_x:.6f}) * {t_y:.6f}       = {w_10:.6f}")
print(f"   w_11 = t_x * t_y             = {t_x:.6f} * {t_y:.6f}             = {w_11:.6f}")
print(f"   Sum of weights = {w_00 + w_01 + w_10 + w_11:.6f} (should be 1.0)")

# Final interpolation
interpolated_value = w_00 * f_00 + w_01 * f_01 + w_10 * f_10 + w_11 * f_11

print(f"\n7. FINAL CALCULATION:")
print(f"   Value = w_00*f_00 + w_01*f_01 + w_10*f_10 + w_11*f_11")
print(f"   Value = {w_00:.6f}*{f_00:5.1f} + {w_01:.6f}*{f_01:5.1f} + {w_10:.6f}*{f_10:5.1f} + {w_11:.6f}*{f_11:5.1f}")
print(f"   Value = {w_00*f_00:8.4f} + {w_01*f_01:8.4f} + {w_10*f_10:8.4f} + {w_11*f_11:8.4f}")
print(f"   Value = {interpolated_value:.6f}")

print(f"\n8. TARGET POINT RELATIVE POSITION:")
print(f"   In X direction (longitude): {t_x*100:.2f}% of the way from {lon_min}° to {lon_max}°")
print(f"   In Y direction (latitude):  {t_y*100:.2f}% of the way from {lat_min}° to {lat_max}°")

print(f"\n{'='*60}")
print(f"FINAL RESULT: TEC = {interpolated_value:.4f}")
print(f"{'='*60}")

## 12/18/2024 - 18:30

In [ ]:
import pandas as pd

date = '2024-12-18'
time = '18:30:00'
timestamp_wanted = f'{date} {time}'

if df_embrace_maps_2024.index.name == 'DATETIME':
    result = df_embrace_maps_2024.loc[timestamp_wanted:timestamp_wanted]
else:
    if 'DATETIME' in df_embrace_maps_2024.columns:
        df_embrace_maps_2024 = df_embrace_maps_2024.set_index('DATETIME')
        result = df_embrace_maps_2024.loc[timestamp_wanted:timestamp_wanted]
    else:
        print("DATETIME column not found!")
        print("Available columns:", df_embrace_maps_2024.columns.tolist())

if result.empty:
    if not isinstance(df_embrace_maps_2024.index, pd.DatetimeIndex):
        df_embrace_maps_2024.index = pd.to_datetime(df_embrace_maps_2024.index)

    timestamp_dt = pd.to_datetime(timestamp_wanted)

    diference = abs((df_embrace_maps_2024.index - timestamp_dt).total_seconds())
    idx_nearest = diference.argmin()

    result = df_embrace_maps_2024.iloc[[idx_nearest]]
    print(f"Exact date not found. Using closest record: {df_embrace_maps_2024.index[idx_nearest]}")

In [ ]:
result

In [ ]:
embrace_maps = np.array(result.iloc[:]['TECMAP'])

In [ ]:
np_embrace_maps = []
for i in range(len(embrace_maps)):
    np_embrace_maps.append(embrace_maps[i])
np_embrace_maps = np.array(np_embrace_maps)

In [ ]:
np_embrace_maps

In [ ]:
np_embrace_maps.shape

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-90, -30, -60, 20),
                 lat_step: float = 2,
                 lon_step: float = 2.5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class embrace(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-90, -30, -60, 20),
                 lat_step: float = 2,
                 lon_step: float = 2.5):
        super().__init__(extent, lat_step, lon_step)


embrace_tec = embrace(extent=(-90, -30, -60, 20), lat_step=2, lon_step=2.5)
embrace_points = embrace_tec.find_nearest_points(lat=-2.59346, lon=-44.21248)
print(embrace_points)

In [ ]:
embrace_points

In [ ]:
embrace_tec.tec_map = np_embrace_maps[0]

print("=" * 60)
print("DETAILED BILINEAR INTERPOLATION ANALYSIS - EMBRACE")
print("=" * 60)

print("\n1. THE 4 CLOSEST POINTS:")
for i, point in enumerate(embrace_points):
    lat, lon, lat_idx, lon_idx = point
    tec_value = embrace_tec.tec_map[lat_idx, lon_idx]
    print(f"   P{i+1}: ({lat:+6.1f}°, {lon:+6.1f}°) - Indexes: [{lat_idx:2d}, {lon_idx:2d}] - TEC: {tec_value:5.1f}")

target_lat = -2.59346
target_lon = -44.21248

print(f"\n2. TARGET POINT: ({target_lat:+6.2f}°, {target_lon:+6.2f}°)")

# Collect values and coordinates
values = []
for point in embrace_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(embrace_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in embrace_points]
lons = [p[1] for p in embrace_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

print(f"\n3. INTERPOLATION RECTANGLE BOUNDS:")
print(f"   Latitude:  {lat_min:+6.1f}° to {lat_max:+6.1f}° (range: {lat_max - lat_min:4.1f}°)")
print(f"   Longitude: {lon_min:+6.1f}° to {lon_max:+6.1f}° (range: {lon_max - lon_min:4.1f}°)")

# Value mapping
point_values = {}
for i, point in enumerate(embrace_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]  # Bottom-left corner
f_01 = point_values[(lat_min, lon_max)]  # Bottom-right corner
f_10 = point_values[(lat_max, lon_min)]  # Top-left corner
f_11 = point_values[(lat_max, lon_max)]  # Top-right corner

print(f"\n4. TEC VALUES MAPPING:")
print(f"   f_00 (lat_min, lon_min) = ({lat_min:+6.1f}°, {lon_min:+6.1f}°) = {f_00:5.1f}")
print(f"   f_01 (lat_min, lon_max) = ({lat_min:+6.1f}°, {lon_max:+6.1f}°) = {f_01:5.1f}")
print(f"   f_10 (lat_max, lon_min) = ({lat_max:+6.1f}°, {lon_min:+6.1f}°) = {f_10:5.1f}")
print(f"   f_11 (lat_max, lon_max) = ({lat_max:+6.1f}°, {lon_max:+6.1f}°) = {f_11:5.1f}")

# Interpolation parameters calculation
t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

print(f"\n5. INTERPOLATION PARAMETERS:")
print(f"   t_x = (target_lon - lon_min) / (lon_max - lon_min)")
print(f"   t_x = ({target_lon:+6.2f} - ({lon_min:+6.1f})) / ({lon_max:+6.1f} - ({lon_min:+6.1f}))")
print(f"   t_x = {target_lon - lon_min:6.2f} / {lon_max - lon_min:4.1f} = {t_x:.6f}")
print(f"   ")
print(f"   t_y = (target_lat - lat_min) / (lat_max - lat_min)")
print(f"   t_y = ({target_lat:+6.2f} - ({lat_min:+6.1f})) / ({lat_max:+6.1f} - ({lat_min:+6.1f}))")
print(f"   t_y = {target_lat - lat_min:6.2f} / {lat_max - lat_min:4.1f} = {t_y:.6f}")

# Weight calculations
w_00 = (1 - t_x) * (1 - t_y)
w_01 = t_x * (1 - t_y)
w_10 = (1 - t_x) * t_y
w_11 = t_x * t_y

print(f"\n6. INTERPOLATION WEIGHTS:")
print(f"   w_00 = (1 - t_x) * (1 - t_y) = (1 - {t_x:.6f}) * (1 - {t_y:.6f}) = {w_00:.6f}")
print(f"   w_01 = t_x * (1 - t_y)       = {t_x:.6f} * (1 - {t_y:.6f})       = {w_01:.6f}")
print(f"   w_10 = (1 - t_x) * t_y       = (1 - {t_x:.6f}) * {t_y:.6f}       = {w_10:.6f}")
print(f"   w_11 = t_x * t_y             = {t_x:.6f} * {t_y:.6f}             = {w_11:.6f}")
print(f"   Sum of weights = {w_00 + w_01 + w_10 + w_11:.6f} (should be 1.0)")

# Final interpolation
interpolated_value = w_00 * f_00 + w_01 * f_01 + w_10 * f_10 + w_11 * f_11

print(f"\n7. FINAL CALCULATION:")
print(f"   Value = w_00*f_00 + w_01*f_01 + w_10*f_10 + w_11*f_11")
print(f"   Value = {w_00:.6f}*{f_00:5.1f} + {w_01:.6f}*{f_01:5.1f} + {w_10:.6f}*{f_10:5.1f} + {w_11:.6f}*{f_11:5.1f}")
print(f"   Value = {w_00*f_00:8.4f} + {w_01*f_01:8.4f} + {w_10*f_10:8.4f} + {w_11*f_11:8.4f}")
print(f"   Value = {interpolated_value:.6f}")

print(f"\n8. TARGET POINT RELATIVE POSITION:")
print(f"   In X direction (longitude): {t_x*100:.2f}% of the way from {lon_min}° to {lon_max}°")
print(f"   In Y direction (latitude):  {t_y*100:.2f}% of the way from {lat_min}° to {lat_max}°")

print(f"\n{'='*60}")
print(f"FINAL RESULT: TEC = {interpolated_value:.4f}")
print(f"{'='*60}")

## 12/18/2024 - 18:00

In [ ]:
import pandas as pd

date = '2024-12-18'
time = '18:00:00'
timestamp_wanted = f'{date} {time}'

if df_embrace_maps_2024.index.name == 'DATETIME':
    result = df_embrace_maps_2024.loc[timestamp_wanted:timestamp_wanted]
else:
    if 'DATETIME' in df_embrace_maps_2024.columns:
        df_embrace_maps_2024 = df_embrace_maps_2024.set_index('DATETIME')
        result = df_embrace_maps_2024.loc[timestamp_wanted:timestamp_wanted]
    else:
        print("DATETIME column not found!")
        print("Available columns:", df_embrace_maps_2024.columns.tolist())

if result.empty:
    if not isinstance(df_embrace_maps_2024.index, pd.DatetimeIndex):
        df_embrace_maps_2024.index = pd.to_datetime(df_embrace_maps_2024.index)

    timestamp_dt = pd.to_datetime(timestamp_wanted)

    diference = abs((df_embrace_maps_2024.index - timestamp_dt).total_seconds())
    idx_nearest = diference.argmin()

    result = df_embrace_maps_2024.iloc[[idx_nearest]]
    print(f"Exact date not found. Using closest record: {df_embrace_maps_2024.index[idx_nearest]}")

In [ ]:
result

In [ ]:
embrace_maps = np.array(result.iloc[:]['TECMAP'])

In [ ]:
np_embrace_maps = []
for i in range(len(embrace_maps)):
    np_embrace_maps.append(embrace_maps[i])
np_embrace_maps = np.array(np_embrace_maps)

In [ ]:
np_embrace_maps

In [ ]:
np_embrace_maps.shape

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-90, -30, -60, 20),
                 lat_step: float = 2,
                 lon_step: float = 2.5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class embrace(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-90, -30, -60, 20),
                 lat_step: float = 2,
                 lon_step: float = 2.5):
        super().__init__(extent, lat_step, lon_step)


embrace_tec = embrace(extent=(-90, -30, -60, 20), lat_step=2, lon_step=2.5)
embrace_points = embrace_tec.find_nearest_points(lat=-2.59346, lon=-44.21248)
print(embrace_points)

In [ ]:
embrace_points

In [ ]:
embrace_tec.tec_map = np_embrace_maps[0]

print("=" * 60)
print("DETAILED BILINEAR INTERPOLATION ANALYSIS - EMBRACE")
print("=" * 60)

print("\n1. THE 4 CLOSEST POINTS:")
for i, point in enumerate(embrace_points):
    lat, lon, lat_idx, lon_idx = point
    tec_value = embrace_tec.tec_map[lat_idx, lon_idx]
    print(f"   P{i+1}: ({lat:+6.1f}°, {lon:+6.1f}°) - Indexes: [{lat_idx:2d}, {lon_idx:2d}] - TEC: {tec_value:5.1f}")

target_lat = -2.59346
target_lon = -44.21248

print(f"\n2. TARGET POINT: ({target_lat:+6.2f}°, {target_lon:+6.2f}°)")

# Collect values and coordinates
values = []
for point in embrace_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(embrace_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in embrace_points]
lons = [p[1] for p in embrace_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

print(f"\n3. INTERPOLATION RECTANGLE BOUNDS:")
print(f"   Latitude:  {lat_min:+6.1f}° to {lat_max:+6.1f}° (range: {lat_max - lat_min:4.1f}°)")
print(f"   Longitude: {lon_min:+6.1f}° to {lon_max:+6.1f}° (range: {lon_max - lon_min:4.1f}°)")

# Value mapping
point_values = {}
for i, point in enumerate(embrace_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]  # Bottom-left corner
f_01 = point_values[(lat_min, lon_max)]  # Bottom-right corner
f_10 = point_values[(lat_max, lon_min)]  # Top-left corner
f_11 = point_values[(lat_max, lon_max)]  # Top-right corner

print(f"\n4. TEC VALUES MAPPING:")
print(f"   f_00 (lat_min, lon_min) = ({lat_min:+6.1f}°, {lon_min:+6.1f}°) = {f_00:5.1f}")
print(f"   f_01 (lat_min, lon_max) = ({lat_min:+6.1f}°, {lon_max:+6.1f}°) = {f_01:5.1f}")
print(f"   f_10 (lat_max, lon_min) = ({lat_max:+6.1f}°, {lon_min:+6.1f}°) = {f_10:5.1f}")
print(f"   f_11 (lat_max, lon_max) = ({lat_max:+6.1f}°, {lon_max:+6.1f}°) = {f_11:5.1f}")

# Interpolation parameters calculation
t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

print(f"\n5. INTERPOLATION PARAMETERS:")
print(f"   t_x = (target_lon - lon_min) / (lon_max - lon_min)")
print(f"   t_x = ({target_lon:+6.2f} - ({lon_min:+6.1f})) / ({lon_max:+6.1f} - ({lon_min:+6.1f}))")
print(f"   t_x = {target_lon - lon_min:6.2f} / {lon_max - lon_min:4.1f} = {t_x:.6f}")
print(f"   ")
print(f"   t_y = (target_lat - lat_min) / (lat_max - lat_min)")
print(f"   t_y = ({target_lat:+6.2f} - ({lat_min:+6.1f})) / ({lat_max:+6.1f} - ({lat_min:+6.1f}))")
print(f"   t_y = {target_lat - lat_min:6.2f} / {lat_max - lat_min:4.1f} = {t_y:.6f}")

# Weight calculations
w_00 = (1 - t_x) * (1 - t_y)
w_01 = t_x * (1 - t_y)
w_10 = (1 - t_x) * t_y
w_11 = t_x * t_y

print(f"\n6. INTERPOLATION WEIGHTS:")
print(f"   w_00 = (1 - t_x) * (1 - t_y) = (1 - {t_x:.6f}) * (1 - {t_y:.6f}) = {w_00:.6f}")
print(f"   w_01 = t_x * (1 - t_y)       = {t_x:.6f} * (1 - {t_y:.6f})       = {w_01:.6f}")
print(f"   w_10 = (1 - t_x) * t_y       = (1 - {t_x:.6f}) * {t_y:.6f}       = {w_10:.6f}")
print(f"   w_11 = t_x * t_y             = {t_x:.6f} * {t_y:.6f}             = {w_11:.6f}")
print(f"   Sum of weights = {w_00 + w_01 + w_10 + w_11:.6f} (should be 1.0)")

# Final interpolation
interpolated_value = w_00 * f_00 + w_01 * f_01 + w_10 * f_10 + w_11 * f_11

print(f"\n7. FINAL CALCULATION:")
print(f"   Value = w_00*f_00 + w_01*f_01 + w_10*f_10 + w_11*f_11")
print(f"   Value = {w_00:.6f}*{f_00:5.1f} + {w_01:.6f}*{f_01:5.1f} + {w_10:.6f}*{f_10:5.1f} + {w_11:.6f}*{f_11:5.1f}")
print(f"   Value = {w_00*f_00:8.4f} + {w_01*f_01:8.4f} + {w_10*f_10:8.4f} + {w_11*f_11:8.4f}")
print(f"   Value = {interpolated_value:.6f}")

print(f"\n8. TARGET POINT RELATIVE POSITION:")
print(f"   In X direction (longitude): {t_x*100:.2f}% of the way from {lon_min}° to {lon_max}°")
print(f"   In Y direction (latitude):  {t_y*100:.2f}% of the way from {lat_min}° to {lat_max}°")

print(f"\n{'='*60}")
print(f"FINAL RESULT: TEC = {interpolated_value:.4f}")
print(f"{'='*60}")

## 12/22/2024 - 18:00

In [ ]:
import pandas as pd

date = '2024-12-22'
time = '18:00:00'
timestamp_wanted = f'{date} {time}'

if df_embrace_maps_2024.index.name == 'DATETIME':
    result = df_embrace_maps_2024.loc[timestamp_wanted:timestamp_wanted]
else:
    if 'DATETIME' in df_embrace_maps_2024.columns:
        df_embrace_maps_2024 = df_embrace_maps_2024.set_index('DATETIME')
        result = df_embrace_maps_2024.loc[timestamp_wanted:timestamp_wanted]
    else:
        print("DATETIME column not found!")
        print("Available columns:", df_embrace_maps_2024.columns.tolist())

if result.empty:
    if not isinstance(df_embrace_maps_2024.index, pd.DatetimeIndex):
        df_embrace_maps_2024.index = pd.to_datetime(df_embrace_maps_2024.index)

    timestamp_dt = pd.to_datetime(timestamp_wanted)

    diference = abs((df_embrace_maps_2024.index - timestamp_dt).total_seconds())
    idx_nearest = diference.argmin()

    result = df_embrace_maps_2024.iloc[[idx_nearest]]
    print(f"Exact date not found. Using closest record: {df_embrace_maps_2024.index[idx_nearest]}")

In [ ]:
result

In [ ]:
embrace_maps = np.array(result.iloc[:]['TECMAP'])

In [ ]:
np_embrace_maps = []
for i in range(len(embrace_maps)):
    np_embrace_maps.append(embrace_maps[i])
np_embrace_maps = np.array(np_embrace_maps)

In [ ]:
np_embrace_maps

In [ ]:
np_embrace_maps.shape

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-90, -30, -60, 20),
                 lat_step: float = 2,
                 lon_step: float = 2.5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class embrace(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-90, -30, -60, 20),
                 lat_step: float = 2,
                 lon_step: float = 2.5):
        super().__init__(extent, lat_step, lon_step)


embrace_tec = embrace(extent=(-90, -30, -60, 20), lat_step=2, lon_step=2.5)
embrace_points = embrace_tec.find_nearest_points(lat=-2.59346, lon=-44.21248)
print(embrace_points)

In [ ]:
embrace_points

In [ ]:
embrace_tec.tec_map = np_embrace_maps[0]

print("=" * 60)
print("DETAILED BILINEAR INTERPOLATION ANALYSIS - EMBRACE")
print("=" * 60)

print("\n1. THE 4 CLOSEST POINTS:")
for i, point in enumerate(embrace_points):
    lat, lon, lat_idx, lon_idx = point
    tec_value = embrace_tec.tec_map[lat_idx, lon_idx]
    print(f"   P{i+1}: ({lat:+6.1f}°, {lon:+6.1f}°) - Indexes: [{lat_idx:2d}, {lon_idx:2d}] - TEC: {tec_value:5.1f}")

target_lat = -2.59346
target_lon = -44.21248

print(f"\n2. TARGET POINT: ({target_lat:+6.2f}°, {target_lon:+6.2f}°)")

# Collect values and coordinates
values = []
for point in embrace_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(embrace_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in embrace_points]
lons = [p[1] for p in embrace_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

print(f"\n3. INTERPOLATION RECTANGLE BOUNDS:")
print(f"   Latitude:  {lat_min:+6.1f}° to {lat_max:+6.1f}° (range: {lat_max - lat_min:4.1f}°)")
print(f"   Longitude: {lon_min:+6.1f}° to {lon_max:+6.1f}° (range: {lon_max - lon_min:4.1f}°)")

# Value mapping
point_values = {}
for i, point in enumerate(embrace_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]  # Bottom-left corner
f_01 = point_values[(lat_min, lon_max)]  # Bottom-right corner
f_10 = point_values[(lat_max, lon_min)]  # Top-left corner
f_11 = point_values[(lat_max, lon_max)]  # Top-right corner

print(f"\n4. TEC VALUES MAPPING:")
print(f"   f_00 (lat_min, lon_min) = ({lat_min:+6.1f}°, {lon_min:+6.1f}°) = {f_00:5.1f}")
print(f"   f_01 (lat_min, lon_max) = ({lat_min:+6.1f}°, {lon_max:+6.1f}°) = {f_01:5.1f}")
print(f"   f_10 (lat_max, lon_min) = ({lat_max:+6.1f}°, {lon_min:+6.1f}°) = {f_10:5.1f}")
print(f"   f_11 (lat_max, lon_max) = ({lat_max:+6.1f}°, {lon_max:+6.1f}°) = {f_11:5.1f}")

# Interpolation parameters calculation
t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

print(f"\n5. INTERPOLATION PARAMETERS:")
print(f"   t_x = (target_lon - lon_min) / (lon_max - lon_min)")
print(f"   t_x = ({target_lon:+6.2f} - ({lon_min:+6.1f})) / ({lon_max:+6.1f} - ({lon_min:+6.1f}))")
print(f"   t_x = {target_lon - lon_min:6.2f} / {lon_max - lon_min:4.1f} = {t_x:.6f}")
print(f"   ")
print(f"   t_y = (target_lat - lat_min) / (lat_max - lat_min)")
print(f"   t_y = ({target_lat:+6.2f} - ({lat_min:+6.1f})) / ({lat_max:+6.1f} - ({lat_min:+6.1f}))")
print(f"   t_y = {target_lat - lat_min:6.2f} / {lat_max - lat_min:4.1f} = {t_y:.6f}")

# Weight calculations
w_00 = (1 - t_x) * (1 - t_y)
w_01 = t_x * (1 - t_y)
w_10 = (1 - t_x) * t_y
w_11 = t_x * t_y

print(f"\n6. INTERPOLATION WEIGHTS:")
print(f"   w_00 = (1 - t_x) * (1 - t_y) = (1 - {t_x:.6f}) * (1 - {t_y:.6f}) = {w_00:.6f}")
print(f"   w_01 = t_x * (1 - t_y)       = {t_x:.6f} * (1 - {t_y:.6f})       = {w_01:.6f}")
print(f"   w_10 = (1 - t_x) * t_y       = (1 - {t_x:.6f}) * {t_y:.6f}       = {w_10:.6f}")
print(f"   w_11 = t_x * t_y             = {t_x:.6f} * {t_y:.6f}             = {w_11:.6f}")
print(f"   Sum of weights = {w_00 + w_01 + w_10 + w_11:.6f} (should be 1.0)")

# Final interpolation
interpolated_value = w_00 * f_00 + w_01 * f_01 + w_10 * f_10 + w_11 * f_11

print(f"\n7. FINAL CALCULATION:")
print(f"   Value = w_00*f_00 + w_01*f_01 + w_10*f_10 + w_11*f_11")
print(f"   Value = {w_00:.6f}*{f_00:5.1f} + {w_01:.6f}*{f_01:5.1f} + {w_10:.6f}*{f_10:5.1f} + {w_11:.6f}*{f_11:5.1f}")
print(f"   Value = {w_00*f_00:8.4f} + {w_01*f_01:8.4f} + {w_10*f_10:8.4f} + {w_11*f_11:8.4f}")
print(f"   Value = {interpolated_value:.6f}")

print(f"\n8. TARGET POINT RELATIVE POSITION:")
print(f"   In X direction (longitude): {t_x*100:.2f}% of the way from {lon_min}° to {lon_max}°")
print(f"   In Y direction (latitude):  {t_y*100:.2f}% of the way from {lat_min}° to {lat_max}°")

print(f"\n{'='*60}")
print(f"FINAL RESULT: TEC = {interpolated_value:.4f}")
print(f"{'='*60}")

## 12/19/2024 - 18:00

In [ ]:
import pandas as pd

date = '2024-12-19'
time = '18:00:00'
timestamp_wanted = f'{date} {time}'

if df_embrace_maps_2024.index.name == 'DATETIME':
    result = df_embrace_maps_2024.loc[timestamp_wanted:timestamp_wanted]
else:
    if 'DATETIME' in df_embrace_maps_2024.columns:
        df_embrace_maps_2024 = df_embrace_maps_2024.set_index('DATETIME')
        result = df_embrace_maps_2024.loc[timestamp_wanted:timestamp_wanted]
    else:
        print("DATETIME column not found!")
        print("Available columns:", df_embrace_maps_2024.columns.tolist())

if result.empty:
    if not isinstance(df_embrace_maps_2024.index, pd.DatetimeIndex):
        df_embrace_maps_2024.index = pd.to_datetime(df_embrace_maps_2024.index)

    timestamp_dt = pd.to_datetime(timestamp_wanted)

    diference = abs((df_embrace_maps_2024.index - timestamp_dt).total_seconds())
    idx_nearest = diference.argmin()

    result = df_embrace_maps_2024.iloc[[idx_nearest]]
    print(f"Exact date not found. Using closest record: {df_embrace_maps_2024.index[idx_nearest]}")

In [ ]:
result

In [ ]:
embrace_maps = np.array(result.iloc[:]['TECMAP'])

In [ ]:
np_embrace_maps = []
for i in range(len(embrace_maps)):
    np_embrace_maps.append(embrace_maps[i])
np_embrace_maps = np.array(np_embrace_maps)

In [ ]:
np_embrace_maps

In [ ]:
np_embrace_maps.shape

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-90, -30, -60, 20),
                 lat_step: float = 2,
                 lon_step: float = 2.5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class embrace(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-90, -30, -60, 20),
                 lat_step: float = 2,
                 lon_step: float = 2.5):
        super().__init__(extent, lat_step, lon_step)


embrace_tec = embrace(extent=(-90, -30, -60, 20), lat_step=2, lon_step=2.5)
embrace_points = embrace_tec.find_nearest_points(lat=-2.59346, lon=-44.21248)
print(embrace_points)

In [ ]:
embrace_points

In [ ]:
embrace_tec.tec_map = np_embrace_maps[0]

print("=" * 60)
print("DETAILED BILINEAR INTERPOLATION ANALYSIS - EMBRACE")
print("=" * 60)

print("\n1. THE 4 CLOSEST POINTS:")
for i, point in enumerate(embrace_points):
    lat, lon, lat_idx, lon_idx = point
    tec_value = embrace_tec.tec_map[lat_idx, lon_idx]
    print(f"   P{i+1}: ({lat:+6.1f}°, {lon:+6.1f}°) - Indexes: [{lat_idx:2d}, {lon_idx:2d}] - TEC: {tec_value:5.1f}")

target_lat = -2.59346
target_lon = -44.21248

print(f"\n2. TARGET POINT: ({target_lat:+6.2f}°, {target_lon:+6.2f}°)")

# Collect values and coordinates
values = []
for point in embrace_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(embrace_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in embrace_points]
lons = [p[1] for p in embrace_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

print(f"\n3. INTERPOLATION RECTANGLE BOUNDS:")
print(f"   Latitude:  {lat_min:+6.1f}° to {lat_max:+6.1f}° (range: {lat_max - lat_min:4.1f}°)")
print(f"   Longitude: {lon_min:+6.1f}° to {lon_max:+6.1f}° (range: {lon_max - lon_min:4.1f}°)")

# Value mapping
point_values = {}
for i, point in enumerate(embrace_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]  # Bottom-left corner
f_01 = point_values[(lat_min, lon_max)]  # Bottom-right corner
f_10 = point_values[(lat_max, lon_min)]  # Top-left corner
f_11 = point_values[(lat_max, lon_max)]  # Top-right corner

print(f"\n4. TEC VALUES MAPPING:")
print(f"   f_00 (lat_min, lon_min) = ({lat_min:+6.1f}°, {lon_min:+6.1f}°) = {f_00:5.1f}")
print(f"   f_01 (lat_min, lon_max) = ({lat_min:+6.1f}°, {lon_max:+6.1f}°) = {f_01:5.1f}")
print(f"   f_10 (lat_max, lon_min) = ({lat_max:+6.1f}°, {lon_min:+6.1f}°) = {f_10:5.1f}")
print(f"   f_11 (lat_max, lon_max) = ({lat_max:+6.1f}°, {lon_max:+6.1f}°) = {f_11:5.1f}")

# Interpolation parameters calculation
t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

print(f"\n5. INTERPOLATION PARAMETERS:")
print(f"   t_x = (target_lon - lon_min) / (lon_max - lon_min)")
print(f"   t_x = ({target_lon:+6.2f} - ({lon_min:+6.1f})) / ({lon_max:+6.1f} - ({lon_min:+6.1f}))")
print(f"   t_x = {target_lon - lon_min:6.2f} / {lon_max - lon_min:4.1f} = {t_x:.6f}")
print(f"   ")
print(f"   t_y = (target_lat - lat_min) / (lat_max - lat_min)")
print(f"   t_y = ({target_lat:+6.2f} - ({lat_min:+6.1f})) / ({lat_max:+6.1f} - ({lat_min:+6.1f}))")
print(f"   t_y = {target_lat - lat_min:6.2f} / {lat_max - lat_min:4.1f} = {t_y:.6f}")

# Weight calculations
w_00 = (1 - t_x) * (1 - t_y)
w_01 = t_x * (1 - t_y)
w_10 = (1 - t_x) * t_y
w_11 = t_x * t_y

print(f"\n6. INTERPOLATION WEIGHTS:")
print(f"   w_00 = (1 - t_x) * (1 - t_y) = (1 - {t_x:.6f}) * (1 - {t_y:.6f}) = {w_00:.6f}")
print(f"   w_01 = t_x * (1 - t_y)       = {t_x:.6f} * (1 - {t_y:.6f})       = {w_01:.6f}")
print(f"   w_10 = (1 - t_x) * t_y       = (1 - {t_x:.6f}) * {t_y:.6f}       = {w_10:.6f}")
print(f"   w_11 = t_x * t_y             = {t_x:.6f} * {t_y:.6f}             = {w_11:.6f}")
print(f"   Sum of weights = {w_00 + w_01 + w_10 + w_11:.6f} (should be 1.0)")

# Final interpolation
interpolated_value = w_00 * f_00 + w_01 * f_01 + w_10 * f_10 + w_11 * f_11

print(f"\n7. FINAL CALCULATION:")
print(f"   Value = w_00*f_00 + w_01*f_01 + w_10*f_10 + w_11*f_11")
print(f"   Value = {w_00:.6f}*{f_00:5.1f} + {w_01:.6f}*{f_01:5.1f} + {w_10:.6f}*{f_10:5.1f} + {w_11:.6f}*{f_11:5.1f}")
print(f"   Value = {w_00*f_00:8.4f} + {w_01*f_01:8.4f} + {w_10*f_10:8.4f} + {w_11*f_11:8.4f}")
print(f"   Value = {interpolated_value:.6f}")

print(f"\n8. TARGET POINT RELATIVE POSITION:")
print(f"   In X direction (longitude): {t_x*100:.2f}% of the way from {lon_min}° to {lon_max}°")
print(f"   In Y direction (latitude):  {t_y*100:.2f}% of the way from {lat_min}° to {lat_max}°")

print(f"\n{'='*60}")
print(f"FINAL RESULT: TEC = {interpolated_value:.4f}")
print(f"{'='*60}")

## 12/01/2024 - 18:00

In [ ]:
import pandas as pd

date = '2024-12-01'
time = '18:00:00'
timestamp_wanted = f'{date} {time}'

if df_embrace_maps_2024.index.name == 'DATETIME':
    result = df_embrace_maps_2024.loc[timestamp_wanted:timestamp_wanted]
else:
    if 'DATETIME' in df_embrace_maps_2024.columns:
        df_embrace_maps_2024 = df_embrace_maps_2024.set_index('DATETIME')
        result = df_embrace_maps_2024.loc[timestamp_wanted:timestamp_wanted]
    else:
        print("DATETIME column not found!")
        print("Available columns:", df_embrace_maps_2024.columns.tolist())

if result.empty:
    if not isinstance(df_embrace_maps_2024.index, pd.DatetimeIndex):
        df_embrace_maps_2024.index = pd.to_datetime(df_embrace_maps_2024.index)

    timestamp_dt = pd.to_datetime(timestamp_wanted)

    diference = abs((df_embrace_maps_2024.index - timestamp_dt).total_seconds())
    idx_nearest = diference.argmin()

    result = df_embrace_maps_2024.iloc[[idx_nearest]]
    print(f"Exact date not found. Using closest record: {df_embrace_maps_2024.index[idx_nearest]}")

In [ ]:
result

In [ ]:
embrace_maps = np.array(result.iloc[:]['TECMAP'])

In [ ]:
np_embrace_maps = []
for i in range(len(embrace_maps)):
    np_embrace_maps.append(embrace_maps[i])
np_embrace_maps = np.array(np_embrace_maps)

In [ ]:
np_embrace_maps

In [ ]:
np_embrace_maps.shape

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-90, -30, -60, 20),
                 lat_step: float = 2,
                 lon_step: float = 2.5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class embrace(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-90, -30, -60, 20),
                 lat_step: float = 2,
                 lon_step: float = 2.5):
        super().__init__(extent, lat_step, lon_step)


embrace_tec = embrace(extent=(-90, -30, -60, 20), lat_step=2, lon_step=2.5)
embrace_points = embrace_tec.find_nearest_points(lat=-2.59346, lon=-44.21248)
print(embrace_points)

In [ ]:
embrace_points

In [ ]:
embrace_tec.tec_map = np_embrace_maps[0]

print("=" * 60)
print("DETAILED BILINEAR INTERPOLATION ANALYSIS - EMBRACE")
print("=" * 60)

print("\n1. THE 4 CLOSEST POINTS:")
for i, point in enumerate(embrace_points):
    lat, lon, lat_idx, lon_idx = point
    tec_value = embrace_tec.tec_map[lat_idx, lon_idx]
    print(f"   P{i+1}: ({lat:+6.1f}°, {lon:+6.1f}°) - Indexes: [{lat_idx:2d}, {lon_idx:2d}] - TEC: {tec_value:5.1f}")

target_lat = -2.59346
target_lon = -44.21248

print(f"\n2. TARGET POINT: ({target_lat:+6.2f}°, {target_lon:+6.2f}°)")

# Collect values and coordinates
values = []
for point in embrace_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(embrace_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in embrace_points]
lons = [p[1] for p in embrace_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

print(f"\n3. INTERPOLATION RECTANGLE BOUNDS:")
print(f"   Latitude:  {lat_min:+6.1f}° to {lat_max:+6.1f}° (range: {lat_max - lat_min:4.1f}°)")
print(f"   Longitude: {lon_min:+6.1f}° to {lon_max:+6.1f}° (range: {lon_max - lon_min:4.1f}°)")

# Value mapping
point_values = {}
for i, point in enumerate(embrace_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]  # Bottom-left corner
f_01 = point_values[(lat_min, lon_max)]  # Bottom-right corner
f_10 = point_values[(lat_max, lon_min)]  # Top-left corner
f_11 = point_values[(lat_max, lon_max)]  # Top-right corner

print(f"\n4. TEC VALUES MAPPING:")
print(f"   f_00 (lat_min, lon_min) = ({lat_min:+6.1f}°, {lon_min:+6.1f}°) = {f_00:5.1f}")
print(f"   f_01 (lat_min, lon_max) = ({lat_min:+6.1f}°, {lon_max:+6.1f}°) = {f_01:5.1f}")
print(f"   f_10 (lat_max, lon_min) = ({lat_max:+6.1f}°, {lon_min:+6.1f}°) = {f_10:5.1f}")
print(f"   f_11 (lat_max, lon_max) = ({lat_max:+6.1f}°, {lon_max:+6.1f}°) = {f_11:5.1f}")

# Interpolation parameters calculation
t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

print(f"\n5. INTERPOLATION PARAMETERS:")
print(f"   t_x = (target_lon - lon_min) / (lon_max - lon_min)")
print(f"   t_x = ({target_lon:+6.2f} - ({lon_min:+6.1f})) / ({lon_max:+6.1f} - ({lon_min:+6.1f}))")
print(f"   t_x = {target_lon - lon_min:6.2f} / {lon_max - lon_min:4.1f} = {t_x:.6f}")
print(f"   ")
print(f"   t_y = (target_lat - lat_min) / (lat_max - lat_min)")
print(f"   t_y = ({target_lat:+6.2f} - ({lat_min:+6.1f})) / ({lat_max:+6.1f} - ({lat_min:+6.1f}))")
print(f"   t_y = {target_lat - lat_min:6.2f} / {lat_max - lat_min:4.1f} = {t_y:.6f}")

# Weight calculations
w_00 = (1 - t_x) * (1 - t_y)
w_01 = t_x * (1 - t_y)
w_10 = (1 - t_x) * t_y
w_11 = t_x * t_y

print(f"\n6. INTERPOLATION WEIGHTS:")
print(f"   w_00 = (1 - t_x) * (1 - t_y) = (1 - {t_x:.6f}) * (1 - {t_y:.6f}) = {w_00:.6f}")
print(f"   w_01 = t_x * (1 - t_y)       = {t_x:.6f} * (1 - {t_y:.6f})       = {w_01:.6f}")
print(f"   w_10 = (1 - t_x) * t_y       = (1 - {t_x:.6f}) * {t_y:.6f}       = {w_10:.6f}")
print(f"   w_11 = t_x * t_y             = {t_x:.6f} * {t_y:.6f}             = {w_11:.6f}")
print(f"   Sum of weights = {w_00 + w_01 + w_10 + w_11:.6f} (should be 1.0)")

# Final interpolation
interpolated_value = w_00 * f_00 + w_01 * f_01 + w_10 * f_10 + w_11 * f_11

print(f"\n7. FINAL CALCULATION:")
print(f"   Value = w_00*f_00 + w_01*f_01 + w_10*f_10 + w_11*f_11")
print(f"   Value = {w_00:.6f}*{f_00:5.1f} + {w_01:.6f}*{f_01:5.1f} + {w_10:.6f}*{f_10:5.1f} + {w_11:.6f}*{f_11:5.1f}")
print(f"   Value = {w_00*f_00:8.4f} + {w_01*f_01:8.4f} + {w_10*f_10:8.4f} + {w_11*f_11:8.4f}")
print(f"   Value = {interpolated_value:.6f}")

print(f"\n8. TARGET POINT RELATIVE POSITION:")
print(f"   In X direction (longitude): {t_x*100:.2f}% of the way from {lon_min}° to {lon_max}°")
print(f"   In Y direction (latitude):  {t_y*100:.2f}% of the way from {lat_min}° to {lat_max}°")

print(f"\n{'='*60}")
print(f"FINAL RESULT: TEC = {interpolated_value:.4f}")
print(f"{'='*60}")

# MAGGIA

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
#Download TF_MAGGIA_TEC_maps_intersection_case_study_from_Sept_05_2024_to_Dec_2024.pkl from https://doi.org/10.5281/zenodo.15453941

In [ ]:
!gdown '1Qi7Xm93lhuwGWQnr3N-gquhzzUxegI33'

In [ ]:
df_maggia_maps_2024 = pd.read_pickle('/content/TF_MAGGIA_TEC_maps_intersection_case_study_from_Sept_05_2024_to_Dec_2024.pkl')

In [ ]:
df_maggia_maps_2024

## 12/18/2024 - 18:20

In [ ]:
import pandas as pd

date = '2024-12-18'
time = '18:20:00'
timestamp_wanted = f'{date} {time}'

result = df_maggia_maps_2024[df_maggia_maps_2024['DATETIME'] == timestamp_wanted]

if result.empty:
    df_temp = df_maggia_maps_2024.copy()
    df_temp['DATETIME'] = pd.to_datetime(df_temp['DATETIME'])
    timestamp_dt = pd.to_datetime(timestamp_wanted)

    df_temp['diff'] = abs((df_temp['DATETIME'] - timestamp_dt).dt.total_seconds())
    idx_nearest = df_temp['diff'].idxmin()

    result = df_maggia_maps_2024.iloc[[idx_nearest]]
    print(f"Exact date not found. Using closest record: {df_maggia_maps_2024.iloc[idx_nearest]['DATETIME']}")

In [ ]:
result

In [ ]:
maggia_maps = np.array(result.iloc[:]['TECMAP'])

In [ ]:
np_maggia_maps = []
for i in range(len(maggia_maps)):
    np_maggia_maps.append(maggia_maps[i])
np_maggia_maps = np.array(np_maggia_maps)

In [ ]:
np_maggia_maps

In [ ]:
np_maggia_maps.shape

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80, 40),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class maggia(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80, 40),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        super().__init__(extent, lat_step, lon_step)


maggia_tec = maggia(extent=(-110, 0, -80, 40), lat_step=0.5, lon_step=0.5)
maggia_points = maggia_tec.find_nearest_points(lat=-2.59346, lon=-44.21248)
print(maggia_points)

In [ ]:
maggia_points

In [ ]:
maggia_tec.tec_map = np_maggia_maps[0]

print("=" * 60)
print("DETAILED BILINEAR INTERPOLATION ANALYSIS - MAGGIA")
print("=" * 60)

print("\n1. THE 4 CLOSEST POINTS:")
for i, point in enumerate(maggia_points):
    lat, lon, lat_idx, lon_idx = point
    tec_value = maggia_tec.tec_map[lat_idx, lon_idx]
    print(f"   P{i+1}: ({lat:+6.1f}°, {lon:+6.1f}°) - Indexes: [{lat_idx:2d}, {lon_idx:2d}] - TEC: {tec_value:5.1f}")

target_lat = -2.59346
target_lon = -44.21248

print(f"\n2. TARGET POINT: ({target_lat:+6.2f}°, {target_lon:+6.2f}°)")

# Collect values and coordinates
values = []
for point in maggia_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(maggia_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in maggia_points]
lons = [p[1] for p in maggia_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

print(f"\n3. INTERPOLATION RECTANGLE BOUNDS:")
print(f"   Latitude:  {lat_min:+6.1f}° to {lat_max:+6.1f}° (range: {lat_max - lat_min:4.1f}°)")
print(f"   Longitude: {lon_min:+6.1f}° to {lon_max:+6.1f}° (range: {lon_max - lon_min:4.1f}°)")

# Value mapping
point_values = {}
for i, point in enumerate(maggia_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]  # Bottom-left corner
f_01 = point_values[(lat_min, lon_max)]  # Bottom-right corner
f_10 = point_values[(lat_max, lon_min)]  # Top-left corner
f_11 = point_values[(lat_max, lon_max)]  # Top-right corner

print(f"\n4. TEC VALUES MAPPING:")
print(f"   f_00 (lat_min, lon_min) = ({lat_min:+6.1f}°, {lon_min:+6.1f}°) = {f_00:5.1f}")
print(f"   f_01 (lat_min, lon_max) = ({lat_min:+6.1f}°, {lon_max:+6.1f}°) = {f_01:5.1f}")
print(f"   f_10 (lat_max, lon_min) = ({lat_max:+6.1f}°, {lon_min:+6.1f}°) = {f_10:5.1f}")
print(f"   f_11 (lat_max, lon_max) = ({lat_max:+6.1f}°, {lon_max:+6.1f}°) = {f_11:5.1f}")

# Interpolation parameters calculation
t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

print(f"\n5. INTERPOLATION PARAMETERS:")
print(f"   t_x = (target_lon - lon_min) / (lon_max - lon_min)")
print(f"   t_x = ({target_lon:+6.2f} - ({lon_min:+6.1f})) / ({lon_max:+6.1f} - ({lon_min:+6.1f}))")
print(f"   t_x = {target_lon - lon_min:6.2f} / {lon_max - lon_min:4.1f} = {t_x:.6f}")
print(f"   ")
print(f"   t_y = (target_lat - lat_min) / (lat_max - lat_min)")
print(f"   t_y = ({target_lat:+6.2f} - ({lat_min:+6.1f})) / ({lat_max:+6.1f} - ({lat_min:+6.1f}))")
print(f"   t_y = {target_lat - lat_min:6.2f} / {lat_max - lat_min:4.1f} = {t_y:.6f}")

# Weight calculations
w_00 = (1 - t_x) * (1 - t_y)
w_01 = t_x * (1 - t_y)
w_10 = (1 - t_x) * t_y
w_11 = t_x * t_y

print(f"\n6. INTERPOLATION WEIGHTS:")
print(f"   w_00 = (1 - t_x) * (1 - t_y) = (1 - {t_x:.6f}) * (1 - {t_y:.6f}) = {w_00:.6f}")
print(f"   w_01 = t_x * (1 - t_y)       = {t_x:.6f} * (1 - {t_y:.6f})       = {w_01:.6f}")
print(f"   w_10 = (1 - t_x) * t_y       = (1 - {t_x:.6f}) * {t_y:.6f}       = {w_10:.6f}")
print(f"   w_11 = t_x * t_y             = {t_x:.6f} * {t_y:.6f}             = {w_11:.6f}")
print(f"   Sum of weights = {w_00 + w_01 + w_10 + w_11:.6f} (should be 1.0)")

# Final interpolation
interpolated_value = w_00 * f_00 + w_01 * f_01 + w_10 * f_10 + w_11 * f_11

print(f"\n7. FINAL CALCULATION:")
print(f"   Value = w_00*f_00 + w_01*f_01 + w_10*f_10 + w_11*f_11")
print(f"   Value = {w_00:.6f}*{f_00:5.1f} + {w_01:.6f}*{f_01:5.1f} + {w_10:.6f}*{f_10:5.1f} + {w_11:.6f}*{f_11:5.1f}")
print(f"   Value = {w_00*f_00:8.4f} + {w_01*f_01:8.4f} + {w_10*f_10:8.4f} + {w_11*f_11:8.4f}")
print(f"   Value = {interpolated_value:.6f}")

print(f"\n8. TARGET POINT RELATIVE POSITION:")
print(f"   In X direction (longitude): {t_x*100:.2f}% of the way from {lon_min}° to {lon_max}°")
print(f"   In Y direction (latitude):  {t_y*100:.2f}% of the way from {lat_min}° to {lat_max}°")

print(f"\n{'='*60}")
print(f"FINAL RESULT: TEC = {interpolated_value:.4f}")
print(f"{'='*60}")

## 12/18/2024 - 18:10

In [ ]:
import pandas as pd

date = '2024-12-18'
time = '18:10:00'
timestamp_wanted = f'{date} {time}'

result = df_maggia_maps_2024[df_maggia_maps_2024['DATETIME'] == timestamp_wanted]

if result.empty:
    df_temp = df_maggia_maps_2024.copy()
    df_temp['DATETIME'] = pd.to_datetime(df_temp['DATETIME'])
    timestamp_dt = pd.to_datetime(timestamp_wanted)

    df_temp['diff'] = abs((df_temp['DATETIME'] - timestamp_dt).dt.total_seconds())
    idx_nearest = df_temp['diff'].idxmin()

    result = df_maggia_maps_2024.iloc[[idx_nearest]]
    print(f"Exact date not found. Using closest record: {df_maggia_maps_2024.iloc[idx_nearest]['DATETIME']}")

In [ ]:
result

In [ ]:
maggia_maps = np.array(result.iloc[:]['TECMAP'])

In [ ]:
np_maggia_maps = []
for i in range(len(maggia_maps)):
    np_maggia_maps.append(maggia_maps[i])
np_maggia_maps = np.array(np_maggia_maps)

In [ ]:
np_maggia_maps

In [ ]:
np_maggia_maps.shape

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80, 40),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class maggia(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80, 40),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        super().__init__(extent, lat_step, lon_step)


maggia_tec = maggia(extent=(-110, 0, -80, 40), lat_step=0.5, lon_step=0.5)
maggia_points = maggia_tec.find_nearest_points(lat=-2.59346, lon=-44.21248)
print(maggia_points)

In [ ]:
maggia_points

In [ ]:
maggia_tec.tec_map = np_maggia_maps[0]

print("=" * 60)
print("DETAILED BILINEAR INTERPOLATION ANALYSIS - MAGGIA")
print("=" * 60)

print("\n1. THE 4 CLOSEST POINTS:")
for i, point in enumerate(maggia_points):
    lat, lon, lat_idx, lon_idx = point
    tec_value = maggia_tec.tec_map[lat_idx, lon_idx]
    print(f"   P{i+1}: ({lat:+6.1f}°, {lon:+6.1f}°) - Indexes: [{lat_idx:2d}, {lon_idx:2d}] - TEC: {tec_value:5.1f}")

target_lat = -2.59346
target_lon = -44.21248

print(f"\n2. TARGET POINT: ({target_lat:+6.2f}°, {target_lon:+6.2f}°)")

# Collect values and coordinates
values = []
for point in maggia_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(maggia_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in maggia_points]
lons = [p[1] for p in maggia_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

print(f"\n3. INTERPOLATION RECTANGLE BOUNDS:")
print(f"   Latitude:  {lat_min:+6.1f}° to {lat_max:+6.1f}° (range: {lat_max - lat_min:4.1f}°)")
print(f"   Longitude: {lon_min:+6.1f}° to {lon_max:+6.1f}° (range: {lon_max - lon_min:4.1f}°)")

# Value mapping
point_values = {}
for i, point in enumerate(maggia_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]  # Bottom-left corner
f_01 = point_values[(lat_min, lon_max)]  # Bottom-right corner
f_10 = point_values[(lat_max, lon_min)]  # Top-left corner
f_11 = point_values[(lat_max, lon_max)]  # Top-right corner

print(f"\n4. TEC VALUES MAPPING:")
print(f"   f_00 (lat_min, lon_min) = ({lat_min:+6.1f}°, {lon_min:+6.1f}°) = {f_00:5.1f}")
print(f"   f_01 (lat_min, lon_max) = ({lat_min:+6.1f}°, {lon_max:+6.1f}°) = {f_01:5.1f}")
print(f"   f_10 (lat_max, lon_min) = ({lat_max:+6.1f}°, {lon_min:+6.1f}°) = {f_10:5.1f}")
print(f"   f_11 (lat_max, lon_max) = ({lat_max:+6.1f}°, {lon_max:+6.1f}°) = {f_11:5.1f}")

# Interpolation parameters calculation
t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

print(f"\n5. INTERPOLATION PARAMETERS:")
print(f"   t_x = (target_lon - lon_min) / (lon_max - lon_min)")
print(f"   t_x = ({target_lon:+6.2f} - ({lon_min:+6.1f})) / ({lon_max:+6.1f} - ({lon_min:+6.1f}))")
print(f"   t_x = {target_lon - lon_min:6.2f} / {lon_max - lon_min:4.1f} = {t_x:.6f}")
print(f"   ")
print(f"   t_y = (target_lat - lat_min) / (lat_max - lat_min)")
print(f"   t_y = ({target_lat:+6.2f} - ({lat_min:+6.1f})) / ({lat_max:+6.1f} - ({lat_min:+6.1f}))")
print(f"   t_y = {target_lat - lat_min:6.2f} / {lat_max - lat_min:4.1f} = {t_y:.6f}")

# Weight calculations
w_00 = (1 - t_x) * (1 - t_y)
w_01 = t_x * (1 - t_y)
w_10 = (1 - t_x) * t_y
w_11 = t_x * t_y

print(f"\n6. INTERPOLATION WEIGHTS:")
print(f"   w_00 = (1 - t_x) * (1 - t_y) = (1 - {t_x:.6f}) * (1 - {t_y:.6f}) = {w_00:.6f}")
print(f"   w_01 = t_x * (1 - t_y)       = {t_x:.6f} * (1 - {t_y:.6f})       = {w_01:.6f}")
print(f"   w_10 = (1 - t_x) * t_y       = (1 - {t_x:.6f}) * {t_y:.6f}       = {w_10:.6f}")
print(f"   w_11 = t_x * t_y             = {t_x:.6f} * {t_y:.6f}             = {w_11:.6f}")
print(f"   Sum of weights = {w_00 + w_01 + w_10 + w_11:.6f} (should be 1.0)")

# Final interpolation
interpolated_value = w_00 * f_00 + w_01 * f_01 + w_10 * f_10 + w_11 * f_11

print(f"\n7. FINAL CALCULATION:")
print(f"   Value = w_00*f_00 + w_01*f_01 + w_10*f_10 + w_11*f_11")
print(f"   Value = {w_00:.6f}*{f_00:5.1f} + {w_01:.6f}*{f_01:5.1f} + {w_10:.6f}*{f_10:5.1f} + {w_11:.6f}*{f_11:5.1f}")
print(f"   Value = {w_00*f_00:8.4f} + {w_01*f_01:8.4f} + {w_10*f_10:8.4f} + {w_11*f_11:8.4f}")
print(f"   Value = {interpolated_value:.6f}")

print(f"\n8. TARGET POINT RELATIVE POSITION:")
print(f"   In X direction (longitude): {t_x*100:.2f}% of the way from {lon_min}° to {lon_max}°")
print(f"   In Y direction (latitude):  {t_y*100:.2f}% of the way from {lat_min}° to {lat_max}°")

print(f"\n{'='*60}")
print(f"FINAL RESULT: TEC = {interpolated_value:.4f}")
print(f"{'='*60}")

## 12/18/2024 - 18:30

In [ ]:
import pandas as pd

date = '2024-12-18'
time = '18:30:00'
timestamp_wanted = f'{date} {time}'

result = df_maggia_maps_2024[df_maggia_maps_2024['DATETIME'] == timestamp_wanted]

if result.empty:
    df_temp = df_maggia_maps_2024.copy()
    df_temp['DATETIME'] = pd.to_datetime(df_temp['DATETIME'])
    timestamp_dt = pd.to_datetime(timestamp_wanted)

    df_temp['diff'] = abs((df_temp['DATETIME'] - timestamp_dt).dt.total_seconds())
    idx_nearest = df_temp['diff'].idxmin()

    result = df_maggia_maps_2024.iloc[[idx_nearest]]
    print(f"Exact date not found. Using closest record: {df_maggia_maps_2024.iloc[idx_nearest]['DATETIME']}")

In [ ]:
result

In [ ]:
maggia_maps = np.array(result.iloc[:]['TECMAP'])

In [ ]:
np_maggia_maps = []
for i in range(len(maggia_maps)):
    np_maggia_maps.append(maggia_maps[i])
np_maggia_maps = np.array(np_maggia_maps)

In [ ]:
np_maggia_maps

In [ ]:
np_maggia_maps.shape

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80, 40),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class maggia(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80, 40),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        super().__init__(extent, lat_step, lon_step)


maggia_tec = maggia(extent=(-110, 0, -80, 40), lat_step=0.5, lon_step=0.5)
maggia_points = maggia_tec.find_nearest_points(lat=-2.59346, lon=-44.21248)
print(maggia_points)

In [ ]:
maggia_points

In [ ]:
maggia_tec.tec_map = np_maggia_maps[0]

print("=" * 60)
print("DETAILED BILINEAR INTERPOLATION ANALYSIS - MAGGIA")
print("=" * 60)

print("\n1. THE 4 CLOSEST POINTS:")
for i, point in enumerate(maggia_points):
    lat, lon, lat_idx, lon_idx = point
    tec_value = maggia_tec.tec_map[lat_idx, lon_idx]
    print(f"   P{i+1}: ({lat:+6.1f}°, {lon:+6.1f}°) - Indexes: [{lat_idx:2d}, {lon_idx:2d}] - TEC: {tec_value:5.1f}")

target_lat = -2.59346
target_lon = -44.21248

print(f"\n2. TARGET POINT: ({target_lat:+6.2f}°, {target_lon:+6.2f}°)")

# Collect values and coordinates
values = []
for point in maggia_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(maggia_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in maggia_points]
lons = [p[1] for p in maggia_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

print(f"\n3. INTERPOLATION RECTANGLE BOUNDS:")
print(f"   Latitude:  {lat_min:+6.1f}° to {lat_max:+6.1f}° (range: {lat_max - lat_min:4.1f}°)")
print(f"   Longitude: {lon_min:+6.1f}° to {lon_max:+6.1f}° (range: {lon_max - lon_min:4.1f}°)")

# Value mapping
point_values = {}
for i, point in enumerate(maggia_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]  # Bottom-left corner
f_01 = point_values[(lat_min, lon_max)]  # Bottom-right corner
f_10 = point_values[(lat_max, lon_min)]  # Top-left corner
f_11 = point_values[(lat_max, lon_max)]  # Top-right corner

print(f"\n4. TEC VALUES MAPPING:")
print(f"   f_00 (lat_min, lon_min) = ({lat_min:+6.1f}°, {lon_min:+6.1f}°) = {f_00:5.1f}")
print(f"   f_01 (lat_min, lon_max) = ({lat_min:+6.1f}°, {lon_max:+6.1f}°) = {f_01:5.1f}")
print(f"   f_10 (lat_max, lon_min) = ({lat_max:+6.1f}°, {lon_min:+6.1f}°) = {f_10:5.1f}")
print(f"   f_11 (lat_max, lon_max) = ({lat_max:+6.1f}°, {lon_max:+6.1f}°) = {f_11:5.1f}")

# Interpolation parameters calculation
t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

print(f"\n5. INTERPOLATION PARAMETERS:")
print(f"   t_x = (target_lon - lon_min) / (lon_max - lon_min)")
print(f"   t_x = ({target_lon:+6.2f} - ({lon_min:+6.1f})) / ({lon_max:+6.1f} - ({lon_min:+6.1f}))")
print(f"   t_x = {target_lon - lon_min:6.2f} / {lon_max - lon_min:4.1f} = {t_x:.6f}")
print(f"   ")
print(f"   t_y = (target_lat - lat_min) / (lat_max - lat_min)")
print(f"   t_y = ({target_lat:+6.2f} - ({lat_min:+6.1f})) / ({lat_max:+6.1f} - ({lat_min:+6.1f}))")
print(f"   t_y = {target_lat - lat_min:6.2f} / {lat_max - lat_min:4.1f} = {t_y:.6f}")

# Weight calculations
w_00 = (1 - t_x) * (1 - t_y)
w_01 = t_x * (1 - t_y)
w_10 = (1 - t_x) * t_y
w_11 = t_x * t_y

print(f"\n6. INTERPOLATION WEIGHTS:")
print(f"   w_00 = (1 - t_x) * (1 - t_y) = (1 - {t_x:.6f}) * (1 - {t_y:.6f}) = {w_00:.6f}")
print(f"   w_01 = t_x * (1 - t_y)       = {t_x:.6f} * (1 - {t_y:.6f})       = {w_01:.6f}")
print(f"   w_10 = (1 - t_x) * t_y       = (1 - {t_x:.6f}) * {t_y:.6f}       = {w_10:.6f}")
print(f"   w_11 = t_x * t_y             = {t_x:.6f} * {t_y:.6f}             = {w_11:.6f}")
print(f"   Sum of weights = {w_00 + w_01 + w_10 + w_11:.6f} (should be 1.0)")

# Final interpolation
interpolated_value = w_00 * f_00 + w_01 * f_01 + w_10 * f_10 + w_11 * f_11

print(f"\n7. FINAL CALCULATION:")
print(f"   Value = w_00*f_00 + w_01*f_01 + w_10*f_10 + w_11*f_11")
print(f"   Value = {w_00:.6f}*{f_00:5.1f} + {w_01:.6f}*{f_01:5.1f} + {w_10:.6f}*{f_10:5.1f} + {w_11:.6f}*{f_11:5.1f}")
print(f"   Value = {w_00*f_00:8.4f} + {w_01*f_01:8.4f} + {w_10*f_10:8.4f} + {w_11*f_11:8.4f}")
print(f"   Value = {interpolated_value:.6f}")

print(f"\n8. TARGET POINT RELATIVE POSITION:")
print(f"   In X direction (longitude): {t_x*100:.2f}% of the way from {lon_min}° to {lon_max}°")
print(f"   In Y direction (latitude):  {t_y*100:.2f}% of the way from {lat_min}° to {lat_max}°")

print(f"\n{'='*60}")
print(f"FINAL RESULT: TEC = {interpolated_value:.4f}")
print(f"{'='*60}")

## 12/18/2024 - 18:00

In [ ]:
import pandas as pd

date = '2024-12-18'
time = '18:00:00'
timestamp_wanted = f'{date} {time}'

result = df_maggia_maps_2024[df_maggia_maps_2024['DATETIME'] == timestamp_wanted]

if result.empty:
    df_temp = df_maggia_maps_2024.copy()
    df_temp['DATETIME'] = pd.to_datetime(df_temp['DATETIME'])
    timestamp_dt = pd.to_datetime(timestamp_wanted)

    df_temp['diff'] = abs((df_temp['DATETIME'] - timestamp_dt).dt.total_seconds())
    idx_nearest = df_temp['diff'].idxmin()

    result = df_maggia_maps_2024.iloc[[idx_nearest]]
    print(f"Exact date not found. Using closest record: {df_maggia_maps_2024.iloc[idx_nearest]['DATETIME']}")

In [ ]:
result

In [ ]:
maggia_maps = np.array(result.iloc[:]['TECMAP'])

In [ ]:
np_maggia_maps = []
for i in range(len(maggia_maps)):
    np_maggia_maps.append(maggia_maps[i])
np_maggia_maps = np.array(np_maggia_maps)

In [ ]:
np_maggia_maps

In [ ]:
np_maggia_maps.shape

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80, 40),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class maggia(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80, 40),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        super().__init__(extent, lat_step, lon_step)


maggia_tec = maggia(extent=(-110, 0, -80, 40), lat_step=0.5, lon_step=0.5)
maggia_points = maggia_tec.find_nearest_points(lat=-2.59346, lon=-44.21248)
print(maggia_points)

In [ ]:
maggia_points

In [ ]:
maggia_tec.tec_map = np_maggia_maps[0]

print("=" * 60)
print("DETAILED BILINEAR INTERPOLATION ANALYSIS - MAGGIA")
print("=" * 60)

print("\n1. THE 4 CLOSEST POINTS:")
for i, point in enumerate(maggia_points):
    lat, lon, lat_idx, lon_idx = point
    tec_value = maggia_tec.tec_map[lat_idx, lon_idx]
    print(f"   P{i+1}: ({lat:+6.1f}°, {lon:+6.1f}°) - Indexes: [{lat_idx:2d}, {lon_idx:2d}] - TEC: {tec_value:5.1f}")

target_lat = -2.59346
target_lon = -44.21248

print(f"\n2. TARGET POINT: ({target_lat:+6.2f}°, {target_lon:+6.2f}°)")

# Collect values and coordinates
values = []
for point in maggia_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(maggia_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in maggia_points]
lons = [p[1] for p in maggia_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

print(f"\n3. INTERPOLATION RECTANGLE BOUNDS:")
print(f"   Latitude:  {lat_min:+6.1f}° to {lat_max:+6.1f}° (range: {lat_max - lat_min:4.1f}°)")
print(f"   Longitude: {lon_min:+6.1f}° to {lon_max:+6.1f}° (range: {lon_max - lon_min:4.1f}°)")

# Value mapping
point_values = {}
for i, point in enumerate(maggia_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]  # Bottom-left corner
f_01 = point_values[(lat_min, lon_max)]  # Bottom-right corner
f_10 = point_values[(lat_max, lon_min)]  # Top-left corner
f_11 = point_values[(lat_max, lon_max)]  # Top-right corner

print(f"\n4. TEC VALUES MAPPING:")
print(f"   f_00 (lat_min, lon_min) = ({lat_min:+6.1f}°, {lon_min:+6.1f}°) = {f_00:5.1f}")
print(f"   f_01 (lat_min, lon_max) = ({lat_min:+6.1f}°, {lon_max:+6.1f}°) = {f_01:5.1f}")
print(f"   f_10 (lat_max, lon_min) = ({lat_max:+6.1f}°, {lon_min:+6.1f}°) = {f_10:5.1f}")
print(f"   f_11 (lat_max, lon_max) = ({lat_max:+6.1f}°, {lon_max:+6.1f}°) = {f_11:5.1f}")

# Interpolation parameters calculation
t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

print(f"\n5. INTERPOLATION PARAMETERS:")
print(f"   t_x = (target_lon - lon_min) / (lon_max - lon_min)")
print(f"   t_x = ({target_lon:+6.2f} - ({lon_min:+6.1f})) / ({lon_max:+6.1f} - ({lon_min:+6.1f}))")
print(f"   t_x = {target_lon - lon_min:6.2f} / {lon_max - lon_min:4.1f} = {t_x:.6f}")
print(f"   ")
print(f"   t_y = (target_lat - lat_min) / (lat_max - lat_min)")
print(f"   t_y = ({target_lat:+6.2f} - ({lat_min:+6.1f})) / ({lat_max:+6.1f} - ({lat_min:+6.1f}))")
print(f"   t_y = {target_lat - lat_min:6.2f} / {lat_max - lat_min:4.1f} = {t_y:.6f}")

# Weight calculations
w_00 = (1 - t_x) * (1 - t_y)
w_01 = t_x * (1 - t_y)
w_10 = (1 - t_x) * t_y
w_11 = t_x * t_y

print(f"\n6. INTERPOLATION WEIGHTS:")
print(f"   w_00 = (1 - t_x) * (1 - t_y) = (1 - {t_x:.6f}) * (1 - {t_y:.6f}) = {w_00:.6f}")
print(f"   w_01 = t_x * (1 - t_y)       = {t_x:.6f} * (1 - {t_y:.6f})       = {w_01:.6f}")
print(f"   w_10 = (1 - t_x) * t_y       = (1 - {t_x:.6f}) * {t_y:.6f}       = {w_10:.6f}")
print(f"   w_11 = t_x * t_y             = {t_x:.6f} * {t_y:.6f}             = {w_11:.6f}")
print(f"   Sum of weights = {w_00 + w_01 + w_10 + w_11:.6f} (should be 1.0)")

# Final interpolation
interpolated_value = w_00 * f_00 + w_01 * f_01 + w_10 * f_10 + w_11 * f_11

print(f"\n7. FINAL CALCULATION:")
print(f"   Value = w_00*f_00 + w_01*f_01 + w_10*f_10 + w_11*f_11")
print(f"   Value = {w_00:.6f}*{f_00:5.1f} + {w_01:.6f}*{f_01:5.1f} + {w_10:.6f}*{f_10:5.1f} + {w_11:.6f}*{f_11:5.1f}")
print(f"   Value = {w_00*f_00:8.4f} + {w_01*f_01:8.4f} + {w_10*f_10:8.4f} + {w_11*f_11:8.4f}")
print(f"   Value = {interpolated_value:.6f}")

print(f"\n8. TARGET POINT RELATIVE POSITION:")
print(f"   In X direction (longitude): {t_x*100:.2f}% of the way from {lon_min}° to {lon_max}°")
print(f"   In Y direction (latitude):  {t_y*100:.2f}% of the way from {lat_min}° to {lat_max}°")

print(f"\n{'='*60}")
print(f"FINAL RESULT: TEC = {interpolated_value:.4f}")
print(f"{'='*60}")

## 12/22/2024 - 18:00

In [ ]:
import pandas as pd

date = '2024-12-22'
time = '18:00:00'
timestamp_wanted = f'{date} {time}'

result = df_maggia_maps_2024[df_maggia_maps_2024['DATETIME'] == timestamp_wanted]

if result.empty:
    df_temp = df_maggia_maps_2024.copy()
    df_temp['DATETIME'] = pd.to_datetime(df_temp['DATETIME'])
    timestamp_dt = pd.to_datetime(timestamp_wanted)

    df_temp['diff'] = abs((df_temp['DATETIME'] - timestamp_dt).dt.total_seconds())
    idx_nearest = df_temp['diff'].idxmin()

    result = df_maggia_maps_2024.iloc[[idx_nearest]]
    print(f"Exact date not found. Using closest record: {df_maggia_maps_2024.iloc[idx_nearest]['DATETIME']}")

In [ ]:
result

In [ ]:
maggia_maps = np.array(result.iloc[:]['TECMAP'])

In [ ]:
np_maggia_maps = []
for i in range(len(maggia_maps)):
    np_maggia_maps.append(maggia_maps[i])
np_maggia_maps = np.array(np_maggia_maps)

In [ ]:
np_maggia_maps

In [ ]:
np_maggia_maps.shape

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80, 40),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class maggia(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80, 40),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        super().__init__(extent, lat_step, lon_step)


maggia_tec = maggia(extent=(-110, 0, -80, 40), lat_step=0.5, lon_step=0.5)
maggia_points = maggia_tec.find_nearest_points(lat=-2.59346, lon=-44.21248)
print(maggia_points)

In [ ]:
maggia_points

In [ ]:
maggia_tec.tec_map = np_maggia_maps[0]

print("=" * 60)
print("DETAILED BILINEAR INTERPOLATION ANALYSIS - MAGGIA")
print("=" * 60)

print("\n1. THE 4 CLOSEST POINTS:")
for i, point in enumerate(maggia_points):
    lat, lon, lat_idx, lon_idx = point
    tec_value = maggia_tec.tec_map[lat_idx, lon_idx]
    print(f"   P{i+1}: ({lat:+6.1f}°, {lon:+6.1f}°) - Indexes: [{lat_idx:2d}, {lon_idx:2d}] - TEC: {tec_value:5.1f}")

target_lat = -2.59346
target_lon = -44.21248

print(f"\n2. TARGET POINT: ({target_lat:+6.2f}°, {target_lon:+6.2f}°)")

# Collect values and coordinates
values = []
for point in maggia_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(maggia_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in maggia_points]
lons = [p[1] for p in maggia_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

print(f"\n3. INTERPOLATION RECTANGLE BOUNDS:")
print(f"   Latitude:  {lat_min:+6.1f}° to {lat_max:+6.1f}° (range: {lat_max - lat_min:4.1f}°)")
print(f"   Longitude: {lon_min:+6.1f}° to {lon_max:+6.1f}° (range: {lon_max - lon_min:4.1f}°)")

# Value mapping
point_values = {}
for i, point in enumerate(maggia_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]  # Bottom-left corner
f_01 = point_values[(lat_min, lon_max)]  # Bottom-right corner
f_10 = point_values[(lat_max, lon_min)]  # Top-left corner
f_11 = point_values[(lat_max, lon_max)]  # Top-right corner

print(f"\n4. TEC VALUES MAPPING:")
print(f"   f_00 (lat_min, lon_min) = ({lat_min:+6.1f}°, {lon_min:+6.1f}°) = {f_00:5.1f}")
print(f"   f_01 (lat_min, lon_max) = ({lat_min:+6.1f}°, {lon_max:+6.1f}°) = {f_01:5.1f}")
print(f"   f_10 (lat_max, lon_min) = ({lat_max:+6.1f}°, {lon_min:+6.1f}°) = {f_10:5.1f}")
print(f"   f_11 (lat_max, lon_max) = ({lat_max:+6.1f}°, {lon_max:+6.1f}°) = {f_11:5.1f}")

# Interpolation parameters calculation
t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

print(f"\n5. INTERPOLATION PARAMETERS:")
print(f"   t_x = (target_lon - lon_min) / (lon_max - lon_min)")
print(f"   t_x = ({target_lon:+6.2f} - ({lon_min:+6.1f})) / ({lon_max:+6.1f} - ({lon_min:+6.1f}))")
print(f"   t_x = {target_lon - lon_min:6.2f} / {lon_max - lon_min:4.1f} = {t_x:.6f}")
print(f"   ")
print(f"   t_y = (target_lat - lat_min) / (lat_max - lat_min)")
print(f"   t_y = ({target_lat:+6.2f} - ({lat_min:+6.1f})) / ({lat_max:+6.1f} - ({lat_min:+6.1f}))")
print(f"   t_y = {target_lat - lat_min:6.2f} / {lat_max - lat_min:4.1f} = {t_y:.6f}")

# Weight calculations
w_00 = (1 - t_x) * (1 - t_y)
w_01 = t_x * (1 - t_y)
w_10 = (1 - t_x) * t_y
w_11 = t_x * t_y

print(f"\n6. INTERPOLATION WEIGHTS:")
print(f"   w_00 = (1 - t_x) * (1 - t_y) = (1 - {t_x:.6f}) * (1 - {t_y:.6f}) = {w_00:.6f}")
print(f"   w_01 = t_x * (1 - t_y)       = {t_x:.6f} * (1 - {t_y:.6f})       = {w_01:.6f}")
print(f"   w_10 = (1 - t_x) * t_y       = (1 - {t_x:.6f}) * {t_y:.6f}       = {w_10:.6f}")
print(f"   w_11 = t_x * t_y             = {t_x:.6f} * {t_y:.6f}             = {w_11:.6f}")
print(f"   Sum of weights = {w_00 + w_01 + w_10 + w_11:.6f} (should be 1.0)")

# Final interpolation
interpolated_value = w_00 * f_00 + w_01 * f_01 + w_10 * f_10 + w_11 * f_11

print(f"\n7. FINAL CALCULATION:")
print(f"   Value = w_00*f_00 + w_01*f_01 + w_10*f_10 + w_11*f_11")
print(f"   Value = {w_00:.6f}*{f_00:5.1f} + {w_01:.6f}*{f_01:5.1f} + {w_10:.6f}*{f_10:5.1f} + {w_11:.6f}*{f_11:5.1f}")
print(f"   Value = {w_00*f_00:8.4f} + {w_01*f_01:8.4f} + {w_10*f_10:8.4f} + {w_11*f_11:8.4f}")
print(f"   Value = {interpolated_value:.6f}")

print(f"\n8. TARGET POINT RELATIVE POSITION:")
print(f"   In X direction (longitude): {t_x*100:.2f}% of the way from {lon_min}° to {lon_max}°")
print(f"   In Y direction (latitude):  {t_y*100:.2f}% of the way from {lat_min}° to {lat_max}°")

print(f"\n{'='*60}")
print(f"FINAL RESULT: TEC = {interpolated_value:.4f}")
print(f"{'='*60}")

## 12/19/2024 - 18:00

In [ ]:
import pandas as pd

date = '2024-12-19'
time = '18:00:00'
timestamp_wanted = f'{date} {time}'

result = df_maggia_maps_2024[df_maggia_maps_2024['DATETIME'] == timestamp_wanted]

if result.empty:
    df_temp = df_maggia_maps_2024.copy()
    df_temp['DATETIME'] = pd.to_datetime(df_temp['DATETIME'])
    timestamp_dt = pd.to_datetime(timestamp_wanted)

    df_temp['diff'] = abs((df_temp['DATETIME'] - timestamp_dt).dt.total_seconds())
    idx_nearest = df_temp['diff'].idxmin()

    result = df_maggia_maps_2024.iloc[[idx_nearest]]
    print(f"Exact date not found. Using closest record: {df_maggia_maps_2024.iloc[idx_nearest]['DATETIME']}")

In [ ]:
result

In [ ]:
maggia_maps = np.array(result.iloc[:]['TECMAP'])

In [ ]:
np_maggia_maps = []
for i in range(len(maggia_maps)):
    np_maggia_maps.append(maggia_maps[i])
np_maggia_maps = np.array(np_maggia_maps)

In [ ]:
np_maggia_maps

In [ ]:
np_maggia_maps.shape

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80, 40),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class maggia(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80, 40),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        super().__init__(extent, lat_step, lon_step)


maggia_tec = maggia(extent=(-110, 0, -80, 40), lat_step=0.5, lon_step=0.5)
maggia_points = maggia_tec.find_nearest_points(lat=-2.59346, lon=-44.21248)
print(maggia_points)

In [ ]:
maggia_points

In [ ]:
maggia_tec.tec_map = np_maggia_maps[0]

print("=" * 60)
print("DETAILED BILINEAR INTERPOLATION ANALYSIS - MAGGIA")
print("=" * 60)

print("\n1. THE 4 CLOSEST POINTS:")
for i, point in enumerate(maggia_points):
    lat, lon, lat_idx, lon_idx = point
    tec_value = maggia_tec.tec_map[lat_idx, lon_idx]
    print(f"   P{i+1}: ({lat:+6.1f}°, {lon:+6.1f}°) - Indexes: [{lat_idx:2d}, {lon_idx:2d}] - TEC: {tec_value:5.1f}")

target_lat = -2.59346
target_lon = -44.21248

print(f"\n2. TARGET POINT: ({target_lat:+6.2f}°, {target_lon:+6.2f}°)")

# Collect values and coordinates
values = []
for point in maggia_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(maggia_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in maggia_points]
lons = [p[1] for p in maggia_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

print(f"\n3. INTERPOLATION RECTANGLE BOUNDS:")
print(f"   Latitude:  {lat_min:+6.1f}° to {lat_max:+6.1f}° (range: {lat_max - lat_min:4.1f}°)")
print(f"   Longitude: {lon_min:+6.1f}° to {lon_max:+6.1f}° (range: {lon_max - lon_min:4.1f}°)")

# Value mapping
point_values = {}
for i, point in enumerate(maggia_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]  # Bottom-left corner
f_01 = point_values[(lat_min, lon_max)]  # Bottom-right corner
f_10 = point_values[(lat_max, lon_min)]  # Top-left corner
f_11 = point_values[(lat_max, lon_max)]  # Top-right corner

print(f"\n4. TEC VALUES MAPPING:")
print(f"   f_00 (lat_min, lon_min) = ({lat_min:+6.1f}°, {lon_min:+6.1f}°) = {f_00:5.1f}")
print(f"   f_01 (lat_min, lon_max) = ({lat_min:+6.1f}°, {lon_max:+6.1f}°) = {f_01:5.1f}")
print(f"   f_10 (lat_max, lon_min) = ({lat_max:+6.1f}°, {lon_min:+6.1f}°) = {f_10:5.1f}")
print(f"   f_11 (lat_max, lon_max) = ({lat_max:+6.1f}°, {lon_max:+6.1f}°) = {f_11:5.1f}")

# Interpolation parameters calculation
t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

print(f"\n5. INTERPOLATION PARAMETERS:")
print(f"   t_x = (target_lon - lon_min) / (lon_max - lon_min)")
print(f"   t_x = ({target_lon:+6.2f} - ({lon_min:+6.1f})) / ({lon_max:+6.1f} - ({lon_min:+6.1f}))")
print(f"   t_x = {target_lon - lon_min:6.2f} / {lon_max - lon_min:4.1f} = {t_x:.6f}")
print(f"   ")
print(f"   t_y = (target_lat - lat_min) / (lat_max - lat_min)")
print(f"   t_y = ({target_lat:+6.2f} - ({lat_min:+6.1f})) / ({lat_max:+6.1f} - ({lat_min:+6.1f}))")
print(f"   t_y = {target_lat - lat_min:6.2f} / {lat_max - lat_min:4.1f} = {t_y:.6f}")

# Weight calculations
w_00 = (1 - t_x) * (1 - t_y)
w_01 = t_x * (1 - t_y)
w_10 = (1 - t_x) * t_y
w_11 = t_x * t_y

print(f"\n6. INTERPOLATION WEIGHTS:")
print(f"   w_00 = (1 - t_x) * (1 - t_y) = (1 - {t_x:.6f}) * (1 - {t_y:.6f}) = {w_00:.6f}")
print(f"   w_01 = t_x * (1 - t_y)       = {t_x:.6f} * (1 - {t_y:.6f})       = {w_01:.6f}")
print(f"   w_10 = (1 - t_x) * t_y       = (1 - {t_x:.6f}) * {t_y:.6f}       = {w_10:.6f}")
print(f"   w_11 = t_x * t_y             = {t_x:.6f} * {t_y:.6f}             = {w_11:.6f}")
print(f"   Sum of weights = {w_00 + w_01 + w_10 + w_11:.6f} (should be 1.0)")

# Final interpolation
interpolated_value = w_00 * f_00 + w_01 * f_01 + w_10 * f_10 + w_11 * f_11

print(f"\n7. FINAL CALCULATION:")
print(f"   Value = w_00*f_00 + w_01*f_01 + w_10*f_10 + w_11*f_11")
print(f"   Value = {w_00:.6f}*{f_00:5.1f} + {w_01:.6f}*{f_01:5.1f} + {w_10:.6f}*{f_10:5.1f} + {w_11:.6f}*{f_11:5.1f}")
print(f"   Value = {w_00*f_00:8.4f} + {w_01*f_01:8.4f} + {w_10*f_10:8.4f} + {w_11*f_11:8.4f}")
print(f"   Value = {interpolated_value:.6f}")

print(f"\n8. TARGET POINT RELATIVE POSITION:")
print(f"   In X direction (longitude): {t_x*100:.2f}% of the way from {lon_min}° to {lon_max}°")
print(f"   In Y direction (latitude):  {t_y*100:.2f}% of the way from {lat_min}° to {lat_max}°")

print(f"\n{'='*60}")
print(f"FINAL RESULT: TEC = {interpolated_value:.4f}")
print(f"{'='*60}")

## 12/01/2024 - 18:00

In [ ]:
import pandas as pd

date = '2024-12-01'
time = '18:00:00'
timestamp_wanted = f'{date} {time}'

result = df_maggia_maps_2024[df_maggia_maps_2024['DATETIME'] == timestamp_wanted]

if result.empty:
    df_temp = df_maggia_maps_2024.copy()
    df_temp['DATETIME'] = pd.to_datetime(df_temp['DATETIME'])
    timestamp_dt = pd.to_datetime(timestamp_wanted)

    df_temp['diff'] = abs((df_temp['DATETIME'] - timestamp_dt).dt.total_seconds())
    idx_nearest = df_temp['diff'].idxmin()

    result = df_maggia_maps_2024.iloc[[idx_nearest]]
    print(f"Exact date not found. Using closest record: {df_maggia_maps_2024.iloc[idx_nearest]['DATETIME']}")

In [ ]:
result

In [ ]:
maggia_maps = np.array(result.iloc[:]['TECMAP'])

In [ ]:
np_maggia_maps = []
for i in range(len(maggia_maps)):
    np_maggia_maps.append(maggia_maps[i])
np_maggia_maps = np.array(np_maggia_maps)

In [ ]:
np_maggia_maps

In [ ]:
np_maggia_maps.shape

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80, 40),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class maggia(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80, 40),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        super().__init__(extent, lat_step, lon_step)


maggia_tec = maggia(extent=(-110, 0, -80, 40), lat_step=0.5, lon_step=0.5)
maggia_points = maggia_tec.find_nearest_points(lat=-2.59346, lon=-44.21248)
print(maggia_points)

In [ ]:
maggia_points

In [ ]:
maggia_tec.tec_map = np_maggia_maps[0]

print("=" * 60)
print("DETAILED BILINEAR INTERPOLATION ANALYSIS - MAGGIA")
print("=" * 60)

print("\n1. THE 4 CLOSEST POINTS:")
for i, point in enumerate(maggia_points):
    lat, lon, lat_idx, lon_idx = point
    tec_value = maggia_tec.tec_map[lat_idx, lon_idx]
    print(f"   P{i+1}: ({lat:+6.1f}°, {lon:+6.1f}°) - Indexes: [{lat_idx:2d}, {lon_idx:2d}] - TEC: {tec_value:5.1f}")

target_lat = -2.59346
target_lon = -44.21248

print(f"\n2. TARGET POINT: ({target_lat:+6.2f}°, {target_lon:+6.2f}°)")

# Collect values and coordinates
values = []
for point in maggia_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(maggia_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in maggia_points]
lons = [p[1] for p in maggia_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

print(f"\n3. INTERPOLATION RECTANGLE BOUNDS:")
print(f"   Latitude:  {lat_min:+6.1f}° to {lat_max:+6.1f}° (range: {lat_max - lat_min:4.1f}°)")
print(f"   Longitude: {lon_min:+6.1f}° to {lon_max:+6.1f}° (range: {lon_max - lon_min:4.1f}°)")

# Value mapping
point_values = {}
for i, point in enumerate(maggia_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]  # Bottom-left corner
f_01 = point_values[(lat_min, lon_max)]  # Bottom-right corner
f_10 = point_values[(lat_max, lon_min)]  # Top-left corner
f_11 = point_values[(lat_max, lon_max)]  # Top-right corner

print(f"\n4. TEC VALUES MAPPING:")
print(f"   f_00 (lat_min, lon_min) = ({lat_min:+6.1f}°, {lon_min:+6.1f}°) = {f_00:5.1f}")
print(f"   f_01 (lat_min, lon_max) = ({lat_min:+6.1f}°, {lon_max:+6.1f}°) = {f_01:5.1f}")
print(f"   f_10 (lat_max, lon_min) = ({lat_max:+6.1f}°, {lon_min:+6.1f}°) = {f_10:5.1f}")
print(f"   f_11 (lat_max, lon_max) = ({lat_max:+6.1f}°, {lon_max:+6.1f}°) = {f_11:5.1f}")

# Interpolation parameters calculation
t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

print(f"\n5. INTERPOLATION PARAMETERS:")
print(f"   t_x = (target_lon - lon_min) / (lon_max - lon_min)")
print(f"   t_x = ({target_lon:+6.2f} - ({lon_min:+6.1f})) / ({lon_max:+6.1f} - ({lon_min:+6.1f}))")
print(f"   t_x = {target_lon - lon_min:6.2f} / {lon_max - lon_min:4.1f} = {t_x:.6f}")
print(f"   ")
print(f"   t_y = (target_lat - lat_min) / (lat_max - lat_min)")
print(f"   t_y = ({target_lat:+6.2f} - ({lat_min:+6.1f})) / ({lat_max:+6.1f} - ({lat_min:+6.1f}))")
print(f"   t_y = {target_lat - lat_min:6.2f} / {lat_max - lat_min:4.1f} = {t_y:.6f}")

# Weight calculations
w_00 = (1 - t_x) * (1 - t_y)
w_01 = t_x * (1 - t_y)
w_10 = (1 - t_x) * t_y
w_11 = t_x * t_y

print(f"\n6. INTERPOLATION WEIGHTS:")
print(f"   w_00 = (1 - t_x) * (1 - t_y) = (1 - {t_x:.6f}) * (1 - {t_y:.6f}) = {w_00:.6f}")
print(f"   w_01 = t_x * (1 - t_y)       = {t_x:.6f} * (1 - {t_y:.6f})       = {w_01:.6f}")
print(f"   w_10 = (1 - t_x) * t_y       = (1 - {t_x:.6f}) * {t_y:.6f}       = {w_10:.6f}")
print(f"   w_11 = t_x * t_y             = {t_x:.6f} * {t_y:.6f}             = {w_11:.6f}")
print(f"   Sum of weights = {w_00 + w_01 + w_10 + w_11:.6f} (should be 1.0)")

# Final interpolation
interpolated_value = w_00 * f_00 + w_01 * f_01 + w_10 * f_10 + w_11 * f_11

print(f"\n7. FINAL CALCULATION:")
print(f"   Value = w_00*f_00 + w_01*f_01 + w_10*f_10 + w_11*f_11")
print(f"   Value = {w_00:.6f}*{f_00:5.1f} + {w_01:.6f}*{f_01:5.1f} + {w_10:.6f}*{f_10:5.1f} + {w_11:.6f}*{f_11:5.1f}")
print(f"   Value = {w_00*f_00:8.4f} + {w_01*f_01:8.4f} + {w_10*f_10:8.4f} + {w_11*f_11:8.4f}")
print(f"   Value = {interpolated_value:.6f}")

print(f"\n8. TARGET POINT RELATIVE POSITION:")
print(f"   In X direction (longitude): {t_x*100:.2f}% of the way from {lon_min}° to {lon_max}°")
print(f"   In Y direction (latitude):  {t_y*100:.2f}% of the way from {lat_min}° to {lat_max}°")

print(f"\n{'='*60}")
print(f"FINAL RESULT: TEC = {interpolated_value:.4f}")
print(f"{'='*60}")